<a href="https://colab.research.google.com/github/MELES-DS/DATA-SCIENCE-CODES/blob/main/DI4Y_ORG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import io
import os


# =====================================================
# GLOBAL STORAGE
# =====================================================

datasets = {}
dataset_months = {}
sorted_datasets = {}



# =====================================================
# MONTH ORDER
# =====================================================

month_order = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}



# =====================================================
# DETECT MONTH FROM COLUMN HEADERS
# Example:
# July 1, July 2, July 3 ...
# =====================================================

def detect_month_from_dataset(df):

    month_count = {
        month: 0
        for month in month_order
    }


    for column in df.columns:

        column_name = str(column).strip().lower()


        for month in month_order:

            if month.lower() in column_name:

                month_count[month] += 1



    detected_month = max(
        month_count,
        key=month_count.get
    )


    if month_count[detected_month] > 0:

        return detected_month


    return "Unknown"





# =====================================================
# UPLOAD DATASET
# =====================================================

upload = widgets.FileUpload(
    accept=".xlsx,.xls,.xlsm,.xlsb,.csv",
    multiple=True,
    description="📂 Upload Files"
)


display(upload)





# =====================================================
# LOAD FILES
# =====================================================

def load_file(change):

    global datasets
    global dataset_months
    global sorted_datasets


    datasets = {}
    dataset_months = {}
    sorted_datasets = {}



    for key, file_dict in upload.value.items():


        filename = file_dict["metadata"]["name"]


        extension = os.path.splitext(filename)[1].lower()


        file_data = io.BytesIO(
            file_dict["content"]
        )


        try:


            if extension == ".csv":

                df = pd.read_csv(file_data)



            elif extension in [
                ".xlsx",
                ".xls",
                ".xlsm",
                ".xlsb"
            ]:

                df = pd.read_excel(
                    file_data,
                    header=1
                )


            else:

                print(
                    "❌ Unsupported file:",
                    filename
                )

                continue



            # Lowercase column names

            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
                .str.lower()
            )



            datasets[filename] = df



            dataset_months[filename] = (
                detect_month_from_dataset(df)
            )



            print(
                "✅ Loaded:",
                filename,
                "| Month:",
                dataset_months[filename],
                "| Shape:",
                df.shape
            )



        except Exception as e:

            print(
                "❌ Error loading",
                filename,
                ":",
                e
            )



    # Automatic chronological sorting

    sorted_datasets = dict(
        sorted(
            datasets.items(),
            key=lambda x:
            month_order.get(
                dataset_months[x[0]],
                99
            )
        )
    )



    print("\n📅 Automatic chronological order:")


    for filename in sorted_datasets:

        print(
            dataset_months[filename],
            "→",
            filename
        )



upload.observe(
    load_file,
    names="value"
)





# =====================================================
# SELECT MONTH BUTTON
# =====================================================

select_month_button = widgets.Button(
    description="📅 Select Month",
    button_style="info"
)


display(select_month_button)





# =====================================================
# MONTH CHECKBOXES
# =====================================================

month_boxes = {}


month_box_container = widgets.VBox([])



for month in month_order:

    box = widgets.Checkbox(
        value=False,
        description=month
    )

    month_boxes[month] = box



month_box_container.children = list(
    month_boxes.values()
)



month_box_container.layout.display = "none"


display(month_box_container)





# =====================================================
# DISPLAY DATASET BUTTON
# =====================================================

display_button = widgets.Button(
    description="▶ Display Dataset",
    button_style="success"
)


display(display_button)


output = widgets.Output()

display(output)





# =====================================================
# SHOW / HIDE MONTH CHECKBOX
# =====================================================

def show_month_selector(button):

    if month_box_container.layout.display == "none":

        month_box_container.layout.display = "block"

        select_month_button.description = (
            "❌ Hide Month"
        )


    else:

        month_box_container.layout.display = "none"

        select_month_button.description = (
            "📅 Select Month"
        )



select_month_button.on_click(
    show_month_selector
)





# =====================================================
# DISPLAY FUNCTION
# =====================================================

def display_selected(button):

    with output:

        clear_output()



        if not datasets:

            print(
                "⚠ Please upload dataset files first."
            )

            return



        # Selected months in checkbox order

        selected_months = [

            month

            for month, box in month_boxes.items()

            if box.value

        ]



        # -------------------------------------------------
        # No checkbox selected:
        # Use chronological order
        # -------------------------------------------------

        if not selected_months:

            display_order = sorted_datasets



        # -------------------------------------------------
        # Checkbox selected:
        # Use selected month order
        # -------------------------------------------------

        else:


            display_order = {}

            missing_months = []



            for selected_month in selected_months:


                found = False



                for filename, df in sorted_datasets.items():


                    if dataset_months[filename] == selected_month:


                        display_order[filename] = df

                        found = True

                        break



                if not found:

                    missing_months.append(
                        selected_month
                    )



            if missing_months:


                print(
                    "⚠ Select the correct month according to your uploaded file."
                )


                print(
                    "Missing:",
                    ", ".join(missing_months)
                )


                return





        # -------------------------------------------------
        # Display result
        # -------------------------------------------------

        print(
            "📅 Dataset display order:"
        )


        for filename, df in display_order.items():


            print("=" * 80)


            print(
                "📅 Month:",
                dataset_months[filename]
            )


            print(
                "📂 File:",
                filename
            )


            print(
                "📊 Shape:",
                df.shape
            )


            print("=" * 80)


            display(
                df.head()
            )



display_button.on_click(
    display_selected
)

FileUpload(value={}, accept='.xlsx,.xls,.xlsm,.xlsb,.csv', description='📂 Upload Files', multiple=True)

Button(button_style='info', description='📅 Select Month', style=ButtonStyle())

Button(button_style='success', description='▶ Display Dataset', style=ButtonStyle())

Output()

✅ Loaded: DIY Attendance preparation  (7).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation  (6).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation  (5).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation  (4).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation  (3).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation  (2).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation  (1).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation .xlsx | Month: July | Shape: (858, 36)

📅 Automatic chronological order:
July → DIY Attendance preparation  (7).xlsx
July → DIY Attendance preparation  (6).xlsx
July → DIY Attendance preparation  (5).xlsx
July → DIY Attendance preparation  (4).xlsx
July → DIY Attendance preparation  (3).xlsx
July → DIY Attendance preparation  (2).xlsx
July → DIY Attendance preparation  (1).xlsx
July → DIY Attendance 

In [2]:
for i, key in enumerate(datasets.keys(), start=1):
    print(i, key)

1 DIY Attendance preparation  (7).xlsx
2 DIY Attendance preparation  (6).xlsx
3 DIY Attendance preparation  (5).xlsx
4 DIY Attendance preparation  (4).xlsx
5 DIY Attendance preparation  (3).xlsx
6 DIY Attendance preparation  (2).xlsx
7 DIY Attendance preparation  (1).xlsx
8 DIY Attendance preparation .xlsx


In [3]:
for filename, df in datasets.items():
    print(f"Columns for {filename}:")
    print(df.columns)
    print("\n")

Columns for DIY Attendance preparation  (7).xlsx:
Index(['column 1', 'column 2', 'name', 'column2', 'diy-id', 'sex', 'age',
       'column1', 'july 1', 'july 2', 'july 3', 'july 4', 'july 5', 'july 6',
       'july 7', 'july 8', 'july 9', 'july 10', 'july 13', 'july 14',
       'july 15', 'july 16', 'july 17', 'july 20', 'july 21', 'july 22',
       'july 23', 'july 24', 'july 27', 'july 28', 'july 29', 'july 30',
       'july 31', 'column 31', 'unnamed: 34', 'unnamed: 35'],
      dtype='object')


Columns for DIY Attendance preparation  (6).xlsx:
Index(['column 1', 'column 2', 'name', 'column2', 'diy-id', 'sex', 'age',
       'column1', 'july 1', 'july 2', 'july 3', 'july 4', 'july 5', 'july 6',
       'july 7', 'july 8', 'july 9', 'july 10', 'july 13', 'july 14',
       'july 15', 'july 16', 'july 17', 'july 20', 'july 21', 'july 22',
       'july 23', 'july 24', 'july 27', 'july 28', 'july 29', 'july 30',
       'july 31', 'column 31', 'unnamed: 34', 'unnamed: 35'],
      dtype='obj

In [4]:
for name, data in datasets.items():
    print(f"✅ Loaded: {name}")
    print(f"📊 Shape: {data.shape}")
    display(data.head())
    print("\n" + "=" * 80 + "\n")

✅ Loaded: DIY Attendance preparation  (7).xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,...,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
0,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,📲🔲 Online Course,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
1,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,✔
3,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,🌐 Internet
4,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,🏛️📚 E - Library




✅ Loaded: DIY Attendance preparation  (6).xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,...,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
0,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,📲🔲 Online Course,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
1,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,✔
3,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,🌐 Internet
4,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,🏛️📚 E - Library




✅ Loaded: DIY Attendance preparation  (5).xlsx
📊 Shape: (858, 36)


...

,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,...,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
0,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,📲🔲 Online Course,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
1,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,✔
3,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,🌐 Internet
4,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,🏛️📚 E - Library




✅ Loaded: DIY Attendance preparation  (1).xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,...,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
0,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,📲🔲 Online Course,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
1,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,✔
3,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,🌐 Internet
4,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,🏛️📚 E - Library




✅ Loaded: DIY Attendance preparation .xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,...,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
0,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,📲🔲 Online Course,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
1,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,✔
3,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,🌐 Internet
4,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,🏛️📚 E - Library


In [5]:
import pandas as pd
import re

for filename, df in datasets.items():

    print(f"\n📂 Checking: {filename}")

    special_chars = []

    for row_idx in df.index:
        for col in df.columns:

            value = df.at[row_idx, col]

            if pd.notna(value):
                value = str(value)

                # Detect emojis and special Unicode symbols
                if re.search(r'[\U0001F300-\U0001FAFF\u2600-\u26FF\u2700-\u27BF]', value):
                    special_chars.append({
                        "Row": row_idx,
                        "Column": col,
                        "Value": value
                    })

    special_df = pd.DataFrame(special_chars)

    if special_df.empty:
        print("✅ No emojis, icons, or special Unicode symbols found.")
    else:
        print(f"✅ Found {len(special_df)} cells containing emojis/icons/special symbols.")
        display(special_df)


📂 Checking: DIY Attendance preparation  (7).xlsx
✅ Found 104 cells containing emojis/icons/special symbols.


,Row,Column,Value
0,0,july 1,📲🔲 Online Course
1,0,july 13,💻🖥️ Skills Practices
2,0,july 15,💻🖥️ Skills Practices
3,2,unnamed: 35,✔
4,3,unnamed: 35,🌐 Internet
...,...,...,...
99,804,july 7,🏛️📚 E - Library
100,805,july 15,💻🖥️ Skills Practices
101,806,july 15,💻🖥️ Skills Practices
102,808,july 16,📲🔲 Online Course



📂 Checking: DIY Attendance preparation  (6).xlsx
✅ Found 104 cells containing emojis/icons/special symbols.


,Row,Column,Value
0,0,july 1,📲🔲 Online Course
1,0,july 13,💻🖥️ Skills Practices
2,0,july 15,💻🖥️ Skills Practices
3,2,unnamed: 35,✔
4,3,unnamed: 35,🌐 Internet
...,...,...,...
99,804,july 7,🏛️📚 E - Library
100,805,july 15,💻🖥️ Skills Practices
101,806,july 15,💻🖥️ Skills Practices
102,808,july 16,📲🔲 Online Course



📂 Checking: DIY Attendance preparation  (5).xlsx
✅ Found 104 cells containing emojis/icons/special symbols.


...

,Row,Column,Value
0,0,july 1,📲🔲 Online Course
1,0,july 13,💻🖥️ Skills Practices
2,0,july 15,💻🖥️ Skills Practices
3,2,unnamed: 35,✔
4,3,unnamed: 35,🌐 Internet
...,...,...,...
99,804,july 7,🏛️📚 E - Library
100,805,july 15,💻🖥️ Skills Practices
101,806,july 15,💻🖥️ Skills Practices
102,808,july 16,📲🔲 Online Course



📂 Checking: DIY Attendance preparation  (1).xlsx
✅ Found 104 cells containing emojis/icons/special symbols.


,Row,Column,Value
0,0,july 1,📲🔲 Online Course
1,0,july 13,💻🖥️ Skills Practices
2,0,july 15,💻🖥️ Skills Practices
3,2,unnamed: 35,✔
4,3,unnamed: 35,🌐 Internet
...,...,...,...
99,804,july 7,🏛️📚 E - Library
100,805,july 15,💻🖥️ Skills Practices
101,806,july 15,💻🖥️ Skills Practices
102,808,july 16,📲🔲 Online Course



📂 Checking: DIY Attendance preparation .xlsx
✅ Found 104 cells containing emojis/icons/special symbols.


,Row,Column,Value
0,0,july 1,📲🔲 Online Course
1,0,july 13,💻🖥️ Skills Practices
2,0,july 15,💻🖥️ Skills Practices
3,2,unnamed: 35,✔
4,3,unnamed: 35,🌐 Internet
...,...,...,...
99,804,july 7,🏛️📚 E - Library
100,805,july 15,💻🖥️ Skills Practices
101,806,july 15,💻🖥️ Skills Practices
102,808,july 16,📲🔲 Online Course


In [7]:
import re

emoji_pattern = re.compile(
    r'[\U0001F300-\U0001FAFF\u2600-\u26FF\u2700-\u27BF\uFE0F]'
)

for filename, df in datasets.items():

    datasets[filename] = df.map(
        lambda x: emoji_pattern.sub("", str(x)).strip()
        if pd.notna(x)
        else x
    )

    print(f"✅ Removed emojis/icons/special symbols from: {filename}")

✅ Removed emojis/icons/special symbols from: DIY Attendance preparation  (7).xlsx
✅ Removed emojis/icons/special symbols from: DIY Attendance preparation  (6).xlsx
✅ Removed emojis/icons/special symbols from: DIY Attendance preparation  (5).xlsx
✅ Removed emojis/icons/special symbols from: DIY Attendance preparation  (4).xlsx
✅ Removed emojis/icons/special symbols from: DIY Attendance preparation  (3).xlsx
✅ Removed emojis/icons/special symbols from: DIY Attendance preparation  (2).xlsx
✅ Removed emojis/icons/special symbols from: DIY Attendance preparation  (1).xlsx
✅ Removed emojis/icons/special symbols from: DIY Attendance preparation .xlsx


In [8]:
from IPython.display import HTML, display

for filename, df in datasets.items():

    print(f"📂 File: {filename}")
    print(f"📊 Shape: {df.shape}")

    display_df = df.head().copy()

    # Make displayed index start from 1
    display_df.index = range(1, len(display_df) + 1)

    display(
        HTML(
            display_df.to_html(index=True)
        )
    )

    print("\n" + "=" * 100 + "\n")

📂 File: DIY Attendance preparation  (7).xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
1,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
2,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
3,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,
4,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,Internet
5,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,E - Library




📂 File: DIY Attendance preparation  (6).xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
1,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
2,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
3,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,
4,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,Internet
5,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,E - Library




📂 File: DIY Attendance preparation  (5).xlsx
📊 Shape: (858, 36)


...

,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
1,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
2,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
3,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,
4,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,Internet
5,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,E - Library




📂 File: DIY Attendance preparation  (1).xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
1,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
2,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
3,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,
4,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,Internet
5,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,E - Library




📂 File: DIY Attendance preparation .xlsx
📊 Shape: (858, 36)


,column 1,column 2,name,column2,diy-id,sex,age,column1,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31,column 31,unnamed: 34,unnamed: 35
1,NaN,1.0,Dawit Damtew,NaN,DIY 0699,M,16,NaN,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,CheckBox
2,NaN,2.0,Nahom Hayile,NaN,DIY 0722,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
3,NaN,3.0,Abayneh Gebeyaw,NaN,NaN,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,
4,NaN,4.0,Yohans Fasika,NaN,NaN,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,Internet
5,NaN,5.0,Eliyas Alemayehu,NaN,NaN,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,E - Library


In [9]:
drop_columns = [
    "column",
    "id",
    "phone number",
    "unnamed",
    "diy-id"
]

for filename, df in datasets.items():

    datasets[filename] = df.drop(
        columns=[
            col for col in df.columns
            if any(
                keyword.lower() in str(col).lower()
                for keyword in drop_columns
            )
        ],
        errors="ignore"
    ).copy()

    print(f"✅ Deleted matching columns from: {filename}")


✅ Deleted matching columns from: DIY Attendance preparation  (7).xlsx
✅ Deleted matching columns from: DIY Attendance preparation  (6).xlsx
✅ Deleted matching columns from: DIY Attendance preparation  (5).xlsx
✅ Deleted matching columns from: DIY Attendance preparation  (4).xlsx
✅ Deleted matching columns from: DIY Attendance preparation  (3).xlsx
✅ Deleted matching columns from: DIY Attendance preparation  (2).xlsx
✅ Deleted matching columns from: DIY Attendance preparation  (1).xlsx
✅ Deleted matching columns from: DIY Attendance preparation .xlsx


In [10]:
from IPython.display import HTML, display

for filename, df in datasets.items():

    print(f"📂 File: {filename}")
    print(f"📊 Shape: {df.shape}")

    display_df = df.head().copy()

    # Make displayed index start from 1
    display_df.index = range(1, len(display_df) + 1)

    display(
        HTML(
            display_df.to_html(index=True)
        )
    )

    print("\n" + "=" * 100 + "\n")

📂 File: DIY Attendance preparation  (7).xlsx
📊 Shape: (858, 28)


,name,sex,age,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31
1,Dawit Damtew,M,16,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nahom Hayile,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Abayneh Gebeyaw,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Yohans Fasika,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Eliyas Alemayehu,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




📂 File: DIY Attendance preparation  (6).xlsx
📊 Shape: (858, 28)


,name,sex,age,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31
1,Dawit Damtew,M,16,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nahom Hayile,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Abayneh Gebeyaw,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Yohans Fasika,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Eliyas Alemayehu,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




📂 File: DIY Attendance preparation  (5).xlsx
📊 Shape: (858, 28)


...

,name,sex,age,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31
1,Dawit Damtew,M,16,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nahom Hayile,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Abayneh Gebeyaw,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Yohans Fasika,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Eliyas Alemayehu,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




📂 File: DIY Attendance preparation  (1).xlsx
📊 Shape: (858, 28)


,name,sex,age,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31
1,Dawit Damtew,M,16,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nahom Hayile,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Abayneh Gebeyaw,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Yohans Fasika,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Eliyas Alemayehu,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




📂 File: DIY Attendance preparation .xlsx
📊 Shape: (858, 28)


,name,sex,age,july 1,july 2,july 3,july 4,july 5,july 6,july 7,july 8,july 9,july 10,july 13,july 14,july 15,july 16,july 17,july 20,july 21,july 22,july 23,july 24,july 27,july 28,july 29,july 30,july 31
1,Dawit Damtew,M,16,Online Course,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Skills Practices,NaN,Skills Practices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nahom Hayile,M,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Abayneh Gebeyaw,M,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Yohans Fasika,M,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Eliyas Alemayehu,M,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
for filename, df in datasets.items():

    print(f"📂 File: {filename}")
    print("📅 Month:", dataset_months[filename])

    print("Total rows:", len(df))

    print(
        "Non-empty Name values:",
        df["name"].count()
    )

    print(
        "Missing values in Name column:",
        df["name"].isnull().sum()
    )

    print(
        "Number of rows in 'Name' column:",
        df["name"].count()
    )

    print("=" * 100)

📂 File: DIY Attendance preparation  (7).xlsx
📅 Month: July
Total rows: 858
Non-empty Name values: 829
Missing values in Name column: 29
Number of rows in 'Name' column: 829
📂 File: DIY Attendance preparation  (6).xlsx
📅 Month: July
Total rows: 858
Non-empty Name values: 829
Missing values in Name column: 29
Number of rows in 'Name' column: 829
📂 File: DIY Attendance preparation  (5).xlsx
📅 Month: July
Total rows: 858
Non-empty Name values: 829
Missing values in Name column: 29
Number of rows in 'Name' column: 829
📂 File: DIY Attendance preparation  (4).xlsx
📅 Month: July
Total rows: 858
Non-empty Name values: 829
Missing values in Name column: 29
Number of rows in 'Name' column: 829
📂 File: DIY Attendance preparation  (3).xlsx
📅 Month: July
Total rows: 858
Non-empty Name values: 829
Missing values in Name column: 29
Number of rows in 'Name' column: 829
📂 File: DIY Attendance preparation  (2).xlsx
📅 Month: July
Total rows: 858
Non-empty Name values: 829
Missing values in Name column: 29

In [12]:
for filename, df in datasets.items():

    print(f"📂 File: {filename}")
    print("📅 Month:", dataset_months[filename])

    # Find duplicated full names
    duplicates = df[df.duplicated(subset=["name"], keep=False)].copy()

    if duplicates.empty:
        print("✅ No duplicated full names found.")
        print("=" * 100)
        continue


    # Create duplicate summary
    dup_summary = (
        duplicates.groupby("name")
        .agg(
            Sex=("sex", lambda x: list(x)),
            Age=("age", lambda x: list(x)),
            Duplication_Count=("name", "count"),
            Duplicated_Index=("name", lambda x: list(x.index))
        )
        .reset_index()
    )


    # Rename column
    dup_summary = dup_summary.rename(
        columns={
            "name": "Full Name"
        }
    )


    # Reorder columns exactly as required
    dup_summary = dup_summary[
        [
            "Full Name",
            "Sex",
            "Age",
            "Duplication_Count",
            "Duplicated_Index"
        ]
    ]


    # Display index starting from 1
    dup_summary.index = range(
        1,
        len(dup_summary) + 1
    )


    print(
        "Total duplicated participant records:",
        len(duplicates)
    )

    print(
        "Total duplicated participant names:",
        len(dup_summary)
    )


    display(dup_summary)

    print("=" * 100)

📂 File: DIY Attendance preparation  (7).xlsx
📅 Month: July
Total duplicated participant records: 148
Total duplicated participant names: 55


,Full Name,Sex,Age,Duplication_Count,Duplicated_Index
1,Abate Amataw,"[M, M]","[36, 34]",2,"[10, 368]"
2,Abebe Tadese,"[M, M, M]","[23, 30, 45]",3,"[98, 104, 451]"
3,Abenezer Fikade,"[M, M]","[19, 22]",2,"[162, 247]"
4,Abera Kidane,"[M, M]","[19, 18]",2,"[152, 236]"
5,Abreham Tesfu,"[M, M]","[17, 24]",2,"[224, 328]"
6,Adamu Abebaw,"[F, M]","[17, 23]",2,"[574, 668]"
7,Agonafer Selemon,"[M, M]","[20, 19]",2,"[201, 337]"
8,Alemayehu Asfaw,"[M, M]","[25, 25]",2,"[59, 380]"
9,Alemu Tiduneh,"[M, M]","[23, 25]",2,"[20, 542]"
10,Alemu Yilma,"[M, M]","[15, 25]",2,"[255, 298]"


📂 File: DIY Attendance preparation  (6).xlsx
📅 Month: July
Total duplicated participant records: 148
Total duplicated participant names: 55


,Full Name,Sex,Age,Duplication_Count,Duplicated_Index
1,Abate Amataw,"[M, M]","[36, 34]",2,"[10, 368]"
2,Abebe Tadese,"[M, M, M]","[23, 30, 45]",3,"[98, 104, 451]"
3,Abenezer Fikade,"[M, M]","[19, 22]",2,"[162, 247]"
4,Abera Kidane,"[M, M]","[19, 18]",2,"[152, 236]"
5,Abreham Tesfu,"[M, M]","[17, 24]",2,"[224, 328]"
6,Adamu Abebaw,"[F, M]","[17, 23]",2,"[574, 668]"
7,Agonafer Selemon,"[M, M]","[20, 19]",2,"[201, 337]"
8,Alemayehu Asfaw,"[M, M]","[25, 25]",2,"[59, 380]"
9,Alemu Tiduneh,"[M, M]","[23, 25]",2,"[20, 542]"
10,Alemu Yilma,"[M, M]","[15, 25]",2,"[255, 298]"


📂 File: DIY Attendance preparation  (5).xlsx
📅 Month: July
Total duplicated participant records: 148
Total duplicated participant names: 55


...

,Full Name,Sex,Age,Duplication_Count,Duplicated_Index
1,Abate Amataw,"[M, M]","[36, 34]",2,"[10, 368]"
2,Abebe Tadese,"[M, M, M]","[23, 30, 45]",3,"[98, 104, 451]"
3,Abenezer Fikade,"[M, M]","[19, 22]",2,"[162, 247]"
4,Abera Kidane,"[M, M]","[19, 18]",2,"[152, 236]"
5,Abreham Tesfu,"[M, M]","[17, 24]",2,"[224, 328]"
6,Adamu Abebaw,"[F, M]","[17, 23]",2,"[574, 668]"
7,Agonafer Selemon,"[M, M]","[20, 19]",2,"[201, 337]"
8,Alemayehu Asfaw,"[M, M]","[25, 25]",2,"[59, 380]"
9,Alemu Tiduneh,"[M, M]","[23, 25]",2,"[20, 542]"
10,Alemu Yilma,"[M, M]","[15, 25]",2,"[255, 298]"


📂 File: DIY Attendance preparation  (1).xlsx
📅 Month: July
Total duplicated participant records: 148
Total duplicated participant names: 55


,Full Name,Sex,Age,Duplication_Count,Duplicated_Index
1,Abate Amataw,"[M, M]","[36, 34]",2,"[10, 368]"
2,Abebe Tadese,"[M, M, M]","[23, 30, 45]",3,"[98, 104, 451]"
3,Abenezer Fikade,"[M, M]","[19, 22]",2,"[162, 247]"
4,Abera Kidane,"[M, M]","[19, 18]",2,"[152, 236]"
5,Abreham Tesfu,"[M, M]","[17, 24]",2,"[224, 328]"
6,Adamu Abebaw,"[F, M]","[17, 23]",2,"[574, 668]"
7,Agonafer Selemon,"[M, M]","[20, 19]",2,"[201, 337]"
8,Alemayehu Asfaw,"[M, M]","[25, 25]",2,"[59, 380]"
9,Alemu Tiduneh,"[M, M]","[23, 25]",2,"[20, 542]"
10,Alemu Yilma,"[M, M]","[15, 25]",2,"[255, 298]"


📂 File: DIY Attendance preparation .xlsx
📅 Month: July
Total duplicated participant records: 148
Total duplicated participant names: 55


,Full Name,Sex,Age,Duplication_Count,Duplicated_Index
1,Abate Amataw,"[M, M]","[36, 34]",2,"[10, 368]"
2,Abebe Tadese,"[M, M, M]","[23, 30, 45]",3,"[98, 104, 451]"
3,Abenezer Fikade,"[M, M]","[19, 22]",2,"[162, 247]"
4,Abera Kidane,"[M, M]","[19, 18]",2,"[152, 236]"
5,Abreham Tesfu,"[M, M]","[17, 24]",2,"[224, 328]"
6,Adamu Abebaw,"[F, M]","[17, 23]",2,"[574, 668]"
7,Agonafer Selemon,"[M, M]","[20, 19]",2,"[201, 337]"
8,Alemayehu Asfaw,"[M, M]","[25, 25]",2,"[59, 380]"
9,Alemu Tiduneh,"[M, M]","[23, 25]",2,"[20, 542]"
10,Alemu Yilma,"[M, M]","[15, 25]",2,"[255, 298]"


In [13]:
for filename, df in datasets.items():

    # Convert age column to integer
    if "age" in df.columns:

        df["age"] = pd.to_numeric(
            df["age"],
            errors="coerce"
        )

        # Use nullable integer type to keep missing values
        df["age"] = df["age"].astype("Int64")

        datasets[filename] = df

        print(f"✅ Age datatype changed for: {filename}")
        print("Age dtype:", df["age"].dtype)

    else:
        print(f"⚠ Age column not found in: {filename}")


print("\n")


# Display missing value summary for all files

for filename, df in datasets.items():

    print("=" * 100)
    print(f"📂 File: {filename}")
    print("📅 Month:", dataset_months[filename])
    print("=" * 100)

    missing_summary = pd.DataFrame({
        "Column": df.columns,
        "Non-Null Count": df.notnull().sum().values,
        "Missing Count": df.isnull().sum().values,
        "Dtype": df.dtypes.values
    })

    # Reset display index starting from 1
    missing_summary.index = range(
        1,
        len(missing_summary) + 1
    )

    display(missing_summary)

    print("\n")

✅ Age datatype changed for: DIY Attendance preparation  (7).xlsx
Age dtype: Int64
✅ Age datatype changed for: DIY Attendance preparation  (6).xlsx
Age dtype: Int64
✅ Age datatype changed for: DIY Attendance preparation  (5).xlsx
Age dtype: Int64
✅ Age datatype changed for: DIY Attendance preparation  (4).xlsx
Age dtype: Int64
✅ Age datatype changed for: DIY Attendance preparation  (3).xlsx
Age dtype: Int64
✅ Age datatype changed for: DIY Attendance preparation  (2).xlsx
Age dtype: Int64
✅ Age datatype changed for: DIY Attendance preparation  (1).xlsx
Age dtype: Int64
✅ Age datatype changed for: DIY Attendance preparation .xlsx
Age dtype: Int64


📂 File: DIY Attendance preparation  (7).xlsx
📅 Month: July


,Column,Non-Null Count,Missing Count,Dtype
1,name,829,29,object
2,sex,811,47,object
3,age,794,64,Int64
4,july 1,14,844,object
5,july 2,16,842,object
6,july 3,14,844,object
7,july 4,2,856,object
8,july 5,1,857,object
9,july 6,13,845,object
10,july 7,8,850,object




📂 File: DIY Attendance preparation  (6).xlsx
📅 Month: July


,Column,Non-Null Count,Missing Count,Dtype
1,name,829,29,object
2,sex,811,47,object
3,age,794,64,Int64
4,july 1,14,844,object
5,july 2,16,842,object
6,july 3,14,844,object
7,july 4,2,856,object
8,july 5,1,857,object
9,july 6,13,845,object
10,july 7,8,850,object




📂 File: DIY Attendance preparation  (5).xlsx
📅 Month: July


...

,Column,Non-Null Count,Missing Count,Dtype
1,name,829,29,object
2,sex,811,47,object
3,age,794,64,Int64
4,july 1,14,844,object
5,july 2,16,842,object
6,july 3,14,844,object
7,july 4,2,856,object
8,july 5,1,857,object
9,july 6,13,845,object
10,july 7,8,850,object




📂 File: DIY Attendance preparation  (1).xlsx
📅 Month: July


,Column,Non-Null Count,Missing Count,Dtype
1,name,829,29,object
2,sex,811,47,object
3,age,794,64,Int64
4,july 1,14,844,object
5,july 2,16,842,object
6,july 3,14,844,object
7,july 4,2,856,object
8,july 5,1,857,object
9,july 6,13,845,object
10,july 7,8,850,object




📂 File: DIY Attendance preparation .xlsx
📅 Month: July


,Column,Non-Null Count,Missing Count,Dtype
1,name,829,29,object
2,sex,811,47,object
3,age,794,64,Int64
4,july 1,14,844,object
5,july 2,16,842,object
6,july 3,14,844,object
7,july 4,2,856,object
8,july 5,1,857,object
9,july 6,13,845,object
10,july 7,8,850,object


In [14]:
for filename, df in datasets.items():

    print("=" * 100)
    print(f"📂 File: {filename}")
    print("📅 Month:", dataset_months[filename])
    print("=" * 100)

    # Fill all missing values with 0
    datasets[filename] = df.fillna(0)

    # Verify missing values are removed
    missing_check = datasets[filename].isnull().sum()

    print("Missing values after filling:")
    print(missing_check)

    print("\n")

📂 File: DIY Attendance preparation  (7).xlsx
📅 Month: July
Missing values after filling:
name       0
sex        0
age        0
july 1     0
july 2     0
july 3     0
july 4     0
july 5     0
july 6     0
july 7     0
july 8     0
july 9     0
july 10    0
july 13    0
july 14    0
july 15    0
july 16    0
july 17    0
july 20    0
july 21    0
july 22    0
july 23    0
july 24    0
july 27    0
july 28    0
july 29    0
july 30    0
july 31    0
dtype: int64


📂 File: DIY Attendance preparation  (6).xlsx
📅 Month: July
Missing values after filling:
name       0
sex        0
age        0
july 1     0
july 2     0
july 3     0
july 4     0
july 5     0
july 6     0
july 7     0
july 8     0
july 9     0
july 10    0
july 13    0
july 14    0
july 15    0
july 16    0
july 17    0
july 20    0
july 21    0
july 22    0
july 23    0
july 24    0
july 27    0
july 28    0
july 29    0
july 30    0
july 31    0
dtype: int64


📂 File: DIY Attendance preparation  (5).xlsx
📅 Month: July
Missi

In [15]:
import pandas as pd

for filename, df in datasets.items():

    print("=" * 100)
    print(f"📂 File: {filename}")
    print("📅 Month:", dataset_months[filename])
    # Get month name
    month = dataset_months[filename]

    # Select columns containing the detected month name
    month_columns = [
        col for col in df.columns
        if month.lower() in str(col).lower()
    ]

    # Extract values from month columns
    values = (
        df[month_columns]
        .astype(str)
        .apply(lambda x: x.str.strip())
        .values
        .flatten()
    )

    # Convert to Series and remove duplicates
    unique_values = pd.Series(values).drop_duplicates()

    # Remove numeric values
    unique_values = unique_values[
        ~unique_values.str.match(r'^\d+(\.\d+)?$', na=False)
    ]

    # Remove unwanted values
    unique_values = unique_values[
        ~unique_values.isin(
            ["0", "0.0", "nan", "NaN", ""]
        )
    ]

    # Reset index for display
    unique_values = unique_values.reset_index(drop=True)

    print("\n✅ Unique values across each month:")
    for index, value in enumerate(unique_values, start=1):

        print(index, ",", value)

    print("\n")

📂 File: DIY Attendance preparation  (7).xlsx
📅 Month: July

✅ Unique values across each month:
1 , Online Course
2 , Skills Practices
3 , E - Library
4 , ‍ Face to Face Training
5 , Internet


📂 File: DIY Attendance preparation  (6).xlsx
📅 Month: July

✅ Unique values across each month:
1 , Online Course
2 , Skills Practices
3 , E - Library
4 , ‍ Face to Face Training
5 , Internet


📂 File: DIY Attendance preparation  (5).xlsx
📅 Month: July

✅ Unique values across each month:
1 , Online Course
2 , Skills Practices
3 , E - Library
4 , ‍ Face to Face Training
5 , Internet


📂 File: DIY Attendance preparation  (4).xlsx
📅 Month: July

✅ Unique values across each month:
1 , Online Course
2 , Skills Practices
3 , E - Library
4 , ‍ Face to Face Training
5 , Internet


📂 File: DIY Attendance preparation  (3).xlsx
📅 Month: July

✅ Unique values across each month:
1 , Online Course
2 , Skills Practices
3 , E - Library
4 , ‍ Face to Face Training
5 , Internet


📂 File: DIY Attendance preparation 

In [16]:
# ============================================================
# 📊 COMPLETE INTERACTIVE ATTENDANCE DASHBOARD
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, clear_output, HTML

import plotly.express as px


# ============================================================
# BASIC DATA CHECK
# ============================================================

if not datasets:

    print("❌ No datasets found.")

else:

    print(f"✅ {len(datasets)} dataset(s) available for dashboard.")


# ============================================================
# DASHBOARD CSS
# ============================================================

display(
    HTML(
        """
        <style>

        .dashboard-title {
            font-size: 30px;
            font-weight: bold;
            margin-bottom: 20px;
        }

        .section-title {
            font-size: 23px;
            font-weight: bold;
            margin-top: 20px;
            margin-bottom: 15px;
        }

        .kpi-container {
            display: flex;
            gap: 18px;
            margin: 20px 0;
            flex-wrap: wrap;
        }

        .kpi-card {
            min-width: 180px;
            padding: 20px;
            border-radius: 12px;
            text-align: center;
            box-shadow: 0px 3px 10px rgba(0,0,0,0.15);
            background-color: #f8f9fa;
        }

        .kpi-card h2 {
            font-size: 32px;
            margin: 0;
        }

        .kpi-card p {
            font-size: 16px;
            margin-top: 8px;
        }

        </style>
        """
    )
)


# ============================================================
# GLOBAL FUNCTIONS
# ============================================================

def get_current_dataframe():

    filename = file_selector.value

    if filename not in datasets:

        return pd.DataFrame()

    return datasets[filename].copy()


# ============================================================
# NORMALIZE GENDER
# ============================================================

def normalize_gender(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.upper()
        .replace(
            {
                "MALE": "M",
                "FEMALE": "F"
            }
        )
    )


# ============================================================
# GET ACTIVITY COLUMNS
# ============================================================

def get_activity_columns(df):

    activity_columns = []

    for column in df.columns:

        column_text = str(column).strip().lower()

        if column_text in ["name", "sex", "age"]:

            continue

        # Numeric day columns: 1 to 31
        if column_text.isdigit():

            day_number = int(column_text)

            if 1 <= day_number <= 31:

                activity_columns.append(column)

                continue

        # Month-based columns
        if any(
            month.lower() in column_text
            for month in month_order.keys()
        ):

            activity_columns.append(column)

    return activity_columns


# ============================================================
# CONVERT WIDE DATA TO LONG ACTIVITY DATA
# ============================================================

def get_activity_data(df):

    activity_columns = get_activity_columns(df)

    if not activity_columns:

        return pd.DataFrame(
            columns=[
                "Name",
                "Sex",
                "Age",
                "Date",
                "Activity"
            ]
        )

    records = []

    for column in activity_columns:

        temp = df[
            [
                "name",
                "sex",
                "age",
                column
            ]
        ].copy()

        temp = temp.rename(
            columns={
                "name": "Name",
                "sex": "Sex",
                "age": "Age",
                column: "Activity"
            }
        )

        temp["Date"] = str(column)

        records.append(temp)

    activity_df = pd.concat(
        records,
        ignore_index=True
    )

    # Clean activity values
    activity_df["Activity"] = (
        activity_df["Activity"]
        .astype(str)
        .str.strip()
    )

    invalid_values = [
        "",
        "0",
        "0.0",
        "nan",
        "NaN",
        "None",
        "N/A",
        "NA"
    ]

    activity_df = activity_df[
        ~activity_df["Activity"].isin(
            invalid_values
        )
    ].copy()

    activity_df["Age"] = pd.to_numeric(
        activity_df["Age"],
        errors="coerce"
    )

    return activity_df


# ============================================================
# DUPLICATED PARTICIPANTS
# ============================================================

def get_duplicate_summary(df):

    if "name" not in df.columns:

        return pd.DataFrame()

    duplicate_rows = df[
        df.duplicated(
            subset=["name"],
            keep=False
        )
    ].copy()

    if duplicate_rows.empty:

        return pd.DataFrame(
            columns=[
                "Full Name",
                "Sex",
                "Age",
                "Duplication Count",
                "Duplicated Index"
            ]
        )

    duplicate_summary = (

        duplicate_rows

        .groupby("name")

        .agg(

            Sex=(
                "sex",
                lambda x: list(x)
            ),

            Age=(
                "age",
                lambda x: list(x)
            ),

            Duplication_Count=(
                "name",
                "count"
            ),

            Duplicated_Index=(
                "name",
                lambda x: list(x.index)
            )

        )

        .reset_index()

    )

    duplicate_summary = duplicate_summary.rename(
        columns={
            "name": "Full Name",
            "Duplication_Count": "Duplication Count",
            "Duplicated_Index": "Duplicated Index"
        }
    )

    return duplicate_summary[
        [
            "Full Name",
            "Sex",
            "Age",
            "Duplication Count",
            "Duplicated Index"
        ]
    ]


# ============================================================
# UNIQUE PARTICIPANTS
# ============================================================

def get_unique_participants(df):

    if "name" not in df.columns:

        return pd.DataFrame()

    unique_df = df[
        ~df["name"].duplicated(
            keep=False
        )
    ].copy()

    unique_df = unique_df[
        [
            "name",
            "sex",
            "age"
        ]
    ].copy()

    unique_df.columns = [
        "Full Name",
        "Sex",
        "Age"
    ]

    return unique_df


# ============================================================
# NEWLY REGISTERED MEMBERS
# ============================================================

def get_new_members(df):

    if "name" not in df.columns:

        return pd.DataFrame()

    new_members = df[
        ~df["name"].duplicated(
            keep=False
        )
    ].copy()

    new_members = new_members[
        [
            "name",
            "sex",
            "age"
        ]
    ].copy()

    new_members.columns = [
        "Full Name",
        "Sex",
        "Age"
    ]

    return new_members


# ============================================================
# KPI CARDS
# ============================================================

def create_kpi_cards(df):

    total = len(df)

    if (
        not df.empty
        and "sex" in df.columns
    ):

        gender = normalize_gender(
            df["sex"]
        )

        male = (
            gender == "M"
        ).sum()

        female = (
            gender == "F"
        ).sum()

    else:

        male = 0
        female = 0

    return HTML(
        f"""
        <div class="kpi-container">

            <div class="kpi-card">
                <h2>{female}</h2>
                <p>Female</p>
            </div>

            <div class="kpi-card">
                <h2>{male}</h2>
                <p>Male</p>
            </div>

            <div class="kpi-card">
                <h2>{total}</h2>
                <p>Total Participants</p>
            </div>

        </div>
        """
    )


# ============================================================
# MAIN SECTION
# ============================================================

main_section = widgets.ToggleButtons(

    options=[
        "Onboarding",
        "Engagement Graph",
        "Other Report"
    ],

    value="Onboarding",

    layout=widgets.Layout(
        width="100%"
    )
)


# ============================================================
# DATASET SELECTOR
# ============================================================

file_selector = widgets.Dropdown(

    options=list(datasets.keys()),

    description="Dataset:",

    layout=widgets.Layout(
        width="450px"
    )
)


# ============================================================
# ONBOARDING CONTROLS
# ============================================================

onboarding_option = widgets.Dropdown(

    options=[
        "Total Participants",
        "Duplicated Participants",
        "Unique Participants",
        "Newly Registered Members"
    ],

    value="Total Participants",

    description="Option:",

    layout=widgets.Layout(
        width="450px"
    )
)


view_list_button = widgets.Button(

    description="📋 View List",

    button_style="primary",

    layout=widgets.Layout(
        width="180px"
    )
)


onboarding_output = widgets.Output()


# ============================================================
# ENGAGEMENT CONTROLS
# ============================================================

engagement_mode = widgets.Dropdown(

    options=[
        "Age Distribution",
        "Gender",
        "Report Type",
        "Activity / Skills"
    ],

    value="Age Distribution",

    description="Analysis:",

    layout=widgets.Layout(
        width="450px"
    )
)


engagement_gender = widgets.Dropdown(

    options=[
        "Male",
        "Female",
        "Both"
    ],

    value="Both",

    description="Gender:",

    layout=widgets.Layout(
        width="400px"
    )
)


engagement_graph_type = widgets.Dropdown(

    options=[
        "Histogram",
        "Line Graph"
    ],

    value="Histogram",

    description="Graph Type:",

    layout=widgets.Layout(
        width="400px"
    )
)


engagement_report_type = widgets.Dropdown(

    options=[
        "Daily",
        "Weekly",
        "Monthly",
        "Yearly",
        "Other"
    ],

    value="Daily",

    description="Report Type:",

    layout=widgets.Layout(
        width="450px"
    )
)


engagement_activity = widgets.Dropdown(

    options=[
        "Online Course",
        "Internet",
        "Skills Practice",
        "Face to Face Training",
        "E-Library",
        "All Activity / Skills"
    ],

    value="All Activity / Skills",

    description="Activity:",

    layout=widgets.Layout(
        width="550px"
    )
)


engagement_graph_button = widgets.Button(

    description="📊 Generate Graph",

    button_style="primary",

    layout=widgets.Layout(
        width="180px"
    )
)


engagement_output = widgets.Output()


# ============================================================
# OTHER REPORT CONTROLS
# ============================================================

other_gender = widgets.Dropdown(

    options=[
        "Male",
        "Female",
        "Both"
    ],

    value="Both",

    description="Gender:",

    layout=widgets.Layout(
        width="400px"
    )
)


other_activity = widgets.Dropdown(

    options=[
        "Online Course",
        "Internet",
        "Skills Practice",
        "Face to Face Training",
        "E-Library",
        "All Activity / Skills"
    ],

    value="All Activity / Skills",

    description="Activity:",

    layout=widgets.Layout(
        width="550px"
    )
)


search_by = widgets.Dropdown(

    options=[
        "Full Name",
        "Age",
        "Day / Date",
        "Number of Activities Taken"
    ],

    value="Full Name",

    description="Search By:",

    layout=widgets.Layout(
        width="500px"
    )
)


search_box = widgets.Text(

    placeholder="Enter search value...",

    description="Search:",

    layout=widgets.Layout(
        width="500px"
    )
)


other_report_output = widgets.Output()


# ============================================================
# ONBOARDING DISPLAY
# ============================================================

def update_onboarding(change=None):

    with onboarding_output:

        clear_output(wait=True)

        df = get_current_dataframe()

        if df.empty:

            print("❌ No dataset selected.")

            return

        option = onboarding_option.value

        # ----------------------------------------------------
        # TOTAL PARTICIPANTS
        # ----------------------------------------------------

        if option == "Total Participants":

            display(
                create_kpi_cards(df)
            )

            print(
                f"Total Participants: {len(df)}"
            )

        # ----------------------------------------------------
        # DUPLICATED PARTICIPANTS
        # ----------------------------------------------------

        elif option == "Duplicated Participants":

            duplicate_summary = (
                get_duplicate_summary(df)
            )

            if duplicate_summary.empty:

                print(
                    "✅ No duplicated participants found."
                )

                return

            duplicate_names = (
                duplicate_summary[
                    "Full Name"
                ]
                .tolist()
            )

            duplicate_df = df[
                df["name"].isin(
                    duplicate_names
                )
            ]

            display(
                create_kpi_cards(
                    duplicate_df
                )
            )

            print(
                "Duplicated Participants:",
                len(duplicate_summary)
            )

        # ----------------------------------------------------
        # UNIQUE PARTICIPANTS
        # ----------------------------------------------------

        elif option == "Unique Participants":

            unique_df = (
                get_unique_participants(df)
            )

            unique_df_for_kpi = (
                unique_df.rename(
                    columns={
                        "Full Name": "name",
                        "Sex": "sex",
                        "Age": "age"
                    }
                )
            )

            display(
                create_kpi_cards(
                    unique_df_for_kpi
                )
            )

            print(
                "Unique Participants:",
                len(unique_df)
            )

        # ----------------------------------------------------
        # NEWLY REGISTERED MEMBERS
        # ----------------------------------------------------

        elif option == "Newly Registered Members":

            new_members = (
                get_new_members(df)
            )

            new_members_for_kpi = (
                new_members.rename(
                    columns={
                        "Full Name": "name",
                        "Sex": "sex",
                        "Age": "age"
                    }
                )
            )

            display(
                create_kpi_cards(
                    new_members_for_kpi
                )
            )

            print(
                "Newly Registered Members:",
                len(new_members)
            )


# ============================================================
# ONBOARDING VIEW LIST
# ============================================================

def view_onboarding_list(button=None):

    with onboarding_output:

        clear_output(wait=True)

        df = get_current_dataframe()

        if df.empty:

            print("❌ No dataset selected.")

            return

        option = onboarding_option.value

        # ----------------------------------------------------
        # TOTAL PARTICIPANTS
        # ----------------------------------------------------

        if option == "Total Participants":

            table = df[
                [
                    "name",
                    "sex",
                    "age"
                ]
            ].copy()

            table.columns = [
                "Full Name",
                "Sex",
                "Age"
            ]

            table.index = range(
                1,
                len(table) + 1
            )

            display(table)

        # ----------------------------------------------------
        # DUPLICATED PARTICIPANTS
        # ----------------------------------------------------

        elif option == "Duplicated Participants":

            duplicate_summary = (
                get_duplicate_summary(df)
            )

            duplicate_summary.index = range(
                1,
                len(duplicate_summary) + 1
            )

            display(
                duplicate_summary
            )

        # ----------------------------------------------------
        # UNIQUE PARTICIPANTS
        # ----------------------------------------------------

        elif option == "Unique Participants":

            unique_df = (
                get_unique_participants(df)
            )

            unique_df.index = range(
                1,
                len(unique_df) + 1
            )

            display(unique_df)

        # ----------------------------------------------------
        # NEW MEMBERS
        # ----------------------------------------------------

        elif option == "Newly Registered Members":

            new_members = (
                get_new_members(df)
            )

            new_members.index = range(
                1,
                len(new_members) + 1
            )

            display(new_members)


# ============================================================
# ENGAGEMENT GRAPH
# ============================================================

def draw_engagement_graph(button=None):

    with engagement_output:

        clear_output(wait=True)

        df = get_current_dataframe()

        if df.empty:

            print("❌ No dataset selected.")

            return

        # ----------------------------------------------------
        # GENDER FILTER
        # ----------------------------------------------------

        selected_gender = (
            engagement_gender.value
        )

        filtered_df = df.copy()

        if selected_gender != "Both":

            gender_code = (
                "M"
                if selected_gender == "Male"
                else "F"
            )

            filtered_df = filtered_df[
                normalize_gender(
                    filtered_df["sex"]
                )
                == gender_code
            ]

        selected_mode = (
            engagement_mode.value
        )

        graph_type = (
            engagement_graph_type.value
        )

        # ====================================================
        # AGE DISTRIBUTION
        # ====================================================

        if selected_mode == "Age Distribution":

            age_df = filtered_df.copy()

            age_df["age"] = pd.to_numeric(
                age_df["age"],
                errors="coerce"
            )

            age_df = age_df.dropna(
                subset=["age"]
            )

            if age_df.empty:

                print(
                    "❌ No valid age data found."
                )

                return

            if graph_type == "Histogram":

                fig = px.histogram(
                    age_df,
                    x="age",
                    nbins=15,
                    title="Age Distribution Histogram"
                )

            else:

                age_counts = (
                    age_df["age"]
                    .value_counts()
                    .sort_index()
                    .reset_index()
                )

                age_counts.columns = [
                    "Age",
                    "Participants"
                ]

                fig = px.line(
                    age_counts,
                    x="Age",
                    y="Participants",
                    markers=True,
                    title="Age Distribution Line Graph"
                )

            fig.show()

        # ====================================================
        # GENDER
        # ====================================================

        elif selected_mode == "Gender":

            gender_df = (
                normalize_gender(
                    filtered_df["sex"]
                )
                .value_counts()
                .reset_index()
            )

            gender_df.columns = [
                "Gender",
                "Count"
            ]

            gender_df["Gender"] = (
                gender_df["Gender"]
                .replace(
                    {
                        "M": "Male",
                        "F": "Female"
                    }
                )
            )

            if graph_type == "Histogram":

                fig = px.histogram(
                    gender_df,
                    x="Gender",
                    y="Count",
                    text_auto=True,
                    title="Gender Distribution"
                )

            else:

                fig = px.line(
                    gender_df,
                    x="Gender",
                    y="Count",
                    markers=True,
                    text="Count",
                    title="Gender Distribution"
                )

            fig.show()

        # ====================================================
        # REPORT TYPE
        # ====================================================

        elif selected_mode == "Report Type":

            activity_df = (
                get_activity_data(
                    filtered_df
                )
            )

            if activity_df.empty:

                print(
                    "❌ No activity records found."
                )

                return

            report_type = (
                engagement_report_type.value
            )

            # DAILY
            if report_type == "Daily":

                report = (
                    activity_df
                    .groupby("Date")
                    .size()
                    .reset_index(
                        name="Participation"
                    )
                )

                x_column = "Date"

                title = "Daily Participation"

            # WEEKLY
            elif report_type == "Weekly":

                activity_df["Day_Number"] = (
                    activity_df["Date"]
                    .astype(str)
                    .str.extract(
                        r"(\d+)$"
                    )[0]
                    .astype(float)
                )

                activity_df["Week"] = (
                    (
                        activity_df["Day_Number"]
                        - 1
                    )
                    // 7
                    + 1
                )

                report = (
                    activity_df
                    .groupby("Week")
                    .size()
                    .reset_index(
                        name="Participation"
                    )
                )

                x_column = "Week"

                title = "Weekly Participation"

            # MONTHLY
            elif report_type == "Monthly":

                month = dataset_months[
                    file_selector.value
                ]

                report = pd.DataFrame(
                    {
                        "Month": [
                            month
                        ],

                        "Participation": [
                            len(activity_df)
                        ]
                    }
                )

                x_column = "Month"

                title = "Monthly Participation"

            # YEARLY
            elif report_type == "Yearly":

                report = pd.DataFrame(
                    {
                        "Year": [
                            "Current Dataset"
                        ],

                        "Participation": [
                            len(activity_df)
                        ]
                    }
                )

                x_column = "Year"

                title = "Yearly Participation"

            # OTHER
            else:

                report = (
                    activity_df[
                        "Activity"
                    ]
                    .value_counts()
                    .reset_index()
                )

                report.columns = [
                    "Activity",
                    "Participation"
                ]

                x_column = "Activity"

                title = "Activity Participation"

            if graph_type == "Histogram":

                fig = px.histogram(
                    report,
                    x=x_column,
                    y="Participation",
                    text_auto=True,
                    title=title
                )

            else:

                fig = px.line(
                    report,
                    x=x_column,
                    y="Participation",
                    markers=True,
                    text="Participation",
                    title=title
                )

            fig.show()

        # ====================================================
        # ACTIVITY / SKILLS
        # ====================================================

        elif selected_mode == "Activity / Skills":

            activity_df = (
                get_activity_data(
                    filtered_df
                )
            )

            if activity_df.empty:

                print(
                    "❌ No activity records found."
                )

                return

            selected_activity = (
                engagement_activity.value
            )

            # ALL ACTIVITIES
            if (
                selected_activity
                == "All Activity / Skills"
            ):

                activity_counts = (
                    activity_df[
                        "Activity"
                    ]
                    .value_counts()
                    .reset_index()
                )

                activity_counts.columns = [
                    "Activity",
                    "Participation"
                ]

                if graph_type == "Histogram":

                    fig = px.histogram(
                        activity_counts,
                        x="Activity",
                        y="Participation",
                        text_auto=True,
                        title="All Activity / Skills"
                    )

                else:

                    fig = px.line(
                        activity_counts,
                        x="Activity",
                        y="Participation",
                        markers=True,
                        text="Participation",
                        title="All Activity / Skills"
                    )

            # SELECTED ACTIVITY
            else:

                matching_activity = (
                    activity_df[
                        activity_df[
                            "Activity"
                        ]
                        .str.contains(
                            selected_activity,
                            case=False,
                            na=False
                        )
                    ]
                )

                if matching_activity.empty:

                    print(
                        "❌ No matching activity found."
                    )

                    return

                activity_counts = (
                    matching_activity
                    .groupby("Date")
                    .size()
                    .reset_index(
                        name="Participation"
                    )
                )

                if graph_type == "Histogram":

                    fig = px.histogram(
                        activity_counts,
                        x="Date",
                        y="Participation",
                        text_auto=True,
                        title=(
                            selected_activity
                            + " Participation"
                        )
                    )

                else:

                    fig = px.line(
                        activity_counts,
                        x="Date",
                        y="Participation",
                        markers=True,
                        text="Participation",
                        title=(
                            selected_activity
                            + " Participation"
                        )
                    )

            fig.show()


# ============================================================
# OTHER REPORT
# ============================================================

def generate_other_report(change=None):

    with other_report_output:

        clear_output(wait=True)

        df = get_current_dataframe()

        if df.empty:

            print(
                "❌ No dataset selected."
            )

            return

        # ----------------------------------------------------
        # GENDER FILTER
        # ----------------------------------------------------

        selected_gender = (
            other_gender.value
        )

        filtered_df = df.copy()

        if selected_gender != "Both":

            gender_code = (
                "M"
                if selected_gender == "Male"
                else "F"
            )

            filtered_df = filtered_df[
                normalize_gender(
                    filtered_df["sex"]
                )
                == gender_code
            ]

        # ----------------------------------------------------
        # ACTIVITY DATA
        # ----------------------------------------------------

        activity_df = (
            get_activity_data(
                filtered_df
            )
        )

        if activity_df.empty:

            print(
                "❌ No activity records found."
            )

            return

        # ----------------------------------------------------
        # ACTIVITY FILTER
        # ----------------------------------------------------

        selected_activity = (
            other_activity.value
        )

        if (
            selected_activity
            != "All Activity / Skills"
        ):

            activity_df = activity_df[
                activity_df[
                    "Activity"
                ]
                .str.contains(
                    selected_activity,
                    case=False,
                    na=False
                )
            ]

        # ----------------------------------------------------
        # SEARCH
        # ----------------------------------------------------

        search_value = (
            search_box.value
            .strip()
            .lower()
        )

        search_type = (
            search_by.value
        )

        if search_value:

            # FULL NAME
            if search_type == "Full Name":

                activity_df = activity_df[
                    activity_df[
                        "Name"
                    ]
                    .astype(str)
                    .str.lower()
                    .str.contains(
                        search_value,
                        na=False
                    )
                ]

            # AGE
            elif search_type == "Age":

                activity_df = activity_df[
                    activity_df[
                        "Age"
                    ]
                    .astype(str)
                    .str.contains(
                        search_value,
                        na=False
                    )
                ]

            # DAY / DATE
            elif search_type == "Day / Date":

                activity_df = activity_df[
                    activity_df[
                        "Date"
                    ]
                    .astype(str)
                    .str.lower()
                    .str.contains(
                        search_value,
                        na=False
                    )
                ]

            # NUMBER OF ACTIVITIES
            elif (
                search_type
                == "Number of Activities Taken"
            ):

                activity_count = (
                    activity_df
                    .groupby(
                        [
                            "Name",
                            "Sex",
                            "Age"
                        ]
                    )
                    .size()
                    .reset_index(
                        name=(
                            "Number of Activities Taken"
                        )
                    )
                )

                activity_count = (
                    activity_count[
                        activity_count[
                            "Number of Activities Taken"
                        ]
                        .astype(str)
                        .str.contains(
                            search_value,
                            na=False
                        )
                    ]
                )

                activity_count.index = range(
                    1,
                    len(activity_count) + 1
                )

                display(
                    activity_count
                )

                return

        # ----------------------------------------------------
        # FINAL RESULT
        # ----------------------------------------------------

        result = activity_df[
            [
                "Name",
                "Sex",
                "Age",
                "Date",
                "Activity"
            ]
        ].copy()

        result.index = range(
            1,
            len(result) + 1
        )

        display(result)


# ============================================================
# DISPLAY ONBOARDING
# ============================================================

def display_onboarding_section():

    display(
        HTML(
            "<h2 class='section-title'>👥 Onboarding</h2>"
        )
    )

    display(
        widgets.HBox(
            [
                onboarding_option,
                view_list_button
            ]
        )
    )

    display(
        onboarding_output
    )

    update_onboarding()


# ============================================================
# DISPLAY ENGAGEMENT
# ============================================================

def display_engagement_section():

    display(
        HTML(
            "<h2 class='section-title'>📈 Engagement Graph</h2>"
        )
    )

    display(
        widgets.VBox(
            [
                engagement_mode,
                engagement_gender,
                engagement_graph_type,
                engagement_report_type,
                engagement_activity,
                engagement_graph_button
            ]
        )
    )

    display(
        engagement_output
    )


# ============================================================
# DISPLAY OTHER REPORT
# ============================================================

def display_other_report_section():

    display(
        HTML(
            "<h2 class='section-title'>📊 Other Report</h2>"
        )
    )

    display(
        widgets.VBox(
            [
                other_gender,
                other_activity,
                search_by,
                search_box
            ]
        )
    )

    display(
        other_report_output
    )

    generate_other_report()


# ============================================================
# MAIN DASHBOARD
# ============================================================

def display_dashboard(change=None):

    clear_output(wait=True)

    display(
        HTML(
            "<h1 class='dashboard-title'>📊 Attendance Dashboard</h1>"
        )
    )

    display(
        file_selector
    )

    display(
        main_section
    )

    if (
        main_section.value
        == "Onboarding"
    ):

        display_onboarding_section()

    elif (
        main_section.value
        == "Engagement Graph"
    ):

        display_engagement_section()

    elif (
        main_section.value
        == "Other Report"
    ):

        display_other_report_section()


# ============================================================
# EVENT CONNECTIONS
# ============================================================

file_selector.observe(
    display_dashboard,
    names="value"
)

main_section.observe(
    display_dashboard,
    names="value"
)

onboarding_option.observe(
    update_onboarding,
    names="value"
)

view_list_button.on_click(
    view_onboarding_list
)

engagement_graph_button.on_click(
    draw_engagement_graph
)

other_gender.observe(
    generate_other_report,
    names="value"
)

other_activity.observe(
    generate_other_report,
    names="value"
)

search_by.observe(
    generate_other_report,
    names="value"
)

search_box.observe(
    generate_other_report,
    names="value"
)


# ============================================================
# START DASHBOARD
# ============================================================

display_dashboard()

Dropdown(description='Dataset:', layout=Layout(width='450px'), options=('DIY Attendance preparation  (7).xlsx'…

ToggleButtons(index=1, layout=Layout(width='100%'), options=('Onboarding', 'Engagement Graph', 'Other Report')…

Output()

In [17]:
# ============================================================
# 📊 COMPLETE INTERACTIVE ATTENDANCE DASHBOARD
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, clear_output, HTML

import plotly.express as px


# ============================================================
# BASIC DATA CHECK
# ============================================================

if not datasets:

    print("❌ No datasets found.")

else:

    print(
        f"✅ {len(datasets)} dataset(s) available for dashboard."
    )


# ============================================================
# DASHBOARD CSS
# ============================================================

display(
    HTML(
        """
        <style>

        .dashboard-title {
            font-size: 30px;
            font-weight: bold;
            margin-bottom: 20px;
        }

        .section-title {
            font-size: 23px;
            font-weight: bold;
            margin-top: 20px;
            margin-bottom: 15px;
        }

        .kpi-container {
            display: flex;
            gap: 18px;
            margin: 20px 0;
            flex-wrap: wrap;
        }

        .kpi-card {
            min-width: 180px;
            padding: 20px;
            border-radius: 12px;
            text-align: center;
            box-shadow: 0px 3px 10px rgba(0,0,0,0.15);
            background-color: #f8f9fa;
        }

        .kpi-card h2 {
            font-size: 32px;
            margin: 0;
        }

        .kpi-card p {
            font-size: 16px;
            margin-top: 8px;
        }

        </style>
        """
    )
)


# ============================================================
# DATASET ORDERING
# ============================================================

def get_ordered_dataset_names():

    dataset_names = list(datasets.keys())

    month_order_list = [
        "January",
        "February",
        "March",
        "April",
        "May",
        "June",
        "July",
        "August",
        "September",
        "October",
        "November",
        "December"
    ]

    def month_position(name):

        name_text = str(name).lower()

        for index, month in enumerate(
            month_order_list
        ):

            if month.lower() in name_text:

                return index

        return 999

    return sorted(
        dataset_names,
        key=month_position
    )


ordered_dataset_names = (
    get_ordered_dataset_names()
)


# ============================================================
# GLOBAL REPORT TYPE
# ============================================================

report_type_selector = widgets.Dropdown(

    options=[

        "Daily",

        "Weekly",

        "Monthly",

        "2 Months",

        "3 Months",

        "4 Months",

        "5 Months",

        "6 Months",

        "7 Months",

        "8 Months",

        "9 Months",

        "10 Months",

        "11 Months",

        "Yearly"

    ],

    value="Monthly",

    description="Report Type:",

    layout=widgets.Layout(

        width="450px"

    )

)


# ============================================================
# MONTH DATASET SELECTOR
# ============================================================

month_selector = widgets.SelectMultiple(

    options=ordered_dataset_names,

    value=(
        (ordered_dataset_names[0],)
        if ordered_dataset_names
        else ()
    ),

    description="Month Dataset:",

    rows=8,

    layout=widgets.Layout(

        width="650px"

    )

)


# ============================================================
# GLOBAL STATUS OUTPUT
# ============================================================

global_status_output = widgets.Output()


# ============================================================
# VALIDATE SELECTED DATASETS
# ============================================================

def get_required_month_count():

    report_type = (
        report_type_selector.value
    )

    if report_type == "Yearly":

        return 12

    if report_type in [

        "2 Months",

        "3 Months",

        "4 Months",

        "5 Months",

        "6 Months",

        "7 Months",

        "8 Months",

        "9 Months",

        "10 Months",

        "11 Months"

    ]:

        return int(
            report_type.split()[0]
        )

    return None


def validate_selection():

    selected_datasets = (
        list(
            month_selector.value
        )
    )

    report_type = (
        report_type_selector.value
    )

    required_count = (
        get_required_month_count()
    )

    if not selected_datasets:

        return False, (
            "❌ Please select at least one "
            "monthly dataset."
        )

    if required_count is not None:

        if len(selected_datasets) != required_count:

            return False, (

                f"❌ {report_type} requires "

                f"exactly {required_count} "

                f"monthly dataset(s). "

                f"You selected "

                f"{len(selected_datasets)}."

            )

    return True, (

        f"✅ {report_type} report using "

        f"{len(selected_datasets)} "

        f"selected dataset(s)."

    )


# ============================================================
# GET COMBINED SELECTED DATA
# ============================================================

def get_current_dataframe():

    selected_datasets = (
        list(
            month_selector.value
        )
    )

    if not selected_datasets:

        return pd.DataFrame()

    dataframes = []

    for dataset_name in selected_datasets:

        if dataset_name in datasets:

            temp_df = (
                datasets[
                    dataset_name
                ]
                .copy()
            )

            temp_df[
                "_Source_Dataset"
            ] = dataset_name

            dataframes.append(
                temp_df
            )

    if not dataframes:

        return pd.DataFrame()

    combined_df = pd.concat(

        dataframes,

        ignore_index=True

    )

    return combined_df


# ============================================================
# UPDATE GLOBAL STATUS
# ============================================================

def update_global_status(change=None):

    with global_status_output:

        clear_output(
            wait=True
        )

        valid, message = (
            validate_selection()
        )

        print(message)


# ============================================================
# NORMALIZE GENDER
# ============================================================

def normalize_gender(series):

    return (

        series

        .astype(str)

        .str.strip()

        .str.upper()

        .replace(

            {

                "MALE": "M",

                "FEMALE": "F"

            }

        )

    )


# ============================================================
# GET ACTIVITY COLUMNS
# ============================================================

def get_activity_columns(df):

    activity_columns = []

    for column in df.columns:

        column_text = (
            str(column)
            .strip()
            .lower()
        )

        if column_text in [

            "name",

            "sex",

            "age",

            "_source_dataset"

        ]:

            continue

        # Numeric day columns
        if column_text.isdigit():

            day_number = int(
                column_text
            )

            if 1 <= day_number <= 31:

                activity_columns.append(
                    column
                )

                continue

        # Month-based columns
        if any(

            month.lower()
            in column_text

            for month in month_order.keys()

        ):

            activity_columns.append(
                column
            )

    return activity_columns


# ============================================================
# CONVERT WIDE DATA INTO ACTIVITY DATA
# ============================================================

def get_activity_data(df):

    activity_columns = (
        get_activity_columns(df)
    )

    if not activity_columns:

        return pd.DataFrame(

            columns=[

                "Name",

                "Sex",

                "Age",

                "Date",

                "Activity",

                "Source Dataset"

            ]

        )

    records = []

    for column in activity_columns:

        temp = df[

            [

                "name",

                "sex",

                "age",

                column,

                "_Source_Dataset"

            ]

        ].copy()

        temp = temp.rename(

            columns={

                "name": "Name",

                "sex": "Sex",

                "age": "Age",

                column: "Activity",

                "_Source_Dataset":
                    "Source Dataset"

            }

        )

        temp["Date"] = str(
            column
        )

        records.append(
            temp
        )

    activity_df = pd.concat(

        records,

        ignore_index=True

    )

    # Clean activity values
    activity_df["Activity"] = (

        activity_df["Activity"]

        .astype(str)

        .str.strip()

    )

    invalid_values = [

        "",

        "0",

        "0.0",

        "nan",

        "NaN",

        "None",

        "N/A",

        "NA"

    ]

    activity_df = activity_df[

        ~activity_df["Activity"].isin(

            invalid_values

        )

    ].copy()

    activity_df["Age"] = pd.to_numeric(

        activity_df["Age"],

        errors="coerce"

    )

    return activity_df


# ============================================================
# DUPLICATED PARTICIPANTS
# ============================================================

def get_duplicate_summary(df):

    if "name" not in df.columns:

        return pd.DataFrame()

    duplicate_rows = df[

        df.duplicated(

            subset=["name"],

            keep=False

        )

    ].copy()

    if duplicate_rows.empty:

        return pd.DataFrame(

            columns=[

                "Full Name",

                "Sex",

                "Age",

                "Duplication Count",

                "Duplicated Index"

            ]

        )

    duplicate_summary = (

        duplicate_rows

        .groupby("name")

        .agg(

            Sex=(

                "sex",

                lambda x: list(x)

            ),

            Age=(

                "age",

                lambda x: list(x)

            ),

            Duplication_Count=(

                "name",

                "count"

            ),

            Duplicated_Index=(

                "name",

                lambda x: list(x.index)

            )

        )

        .reset_index()

    )

    duplicate_summary = (

        duplicate_summary

        .rename(

            columns={

                "name": "Full Name",

                "Duplication_Count":
                    "Duplication Count",

                "Duplicated_Index":
                    "Duplicated Index"

            }

        )

    )

    return duplicate_summary[

        [

            "Full Name",

            "Sex",

            "Age",

            "Duplication Count",

            "Duplicated Index"

        ]

    ]


# ============================================================
# UNIQUE PARTICIPANTS
# ============================================================

def get_unique_participants(df):

    if "name" not in df.columns:

        return pd.DataFrame()

    unique_df = df[

        ~df["name"].duplicated(

            keep=False

        )

    ].copy()

    unique_df = unique_df[

        [

            "name",

            "sex",

            "age"

        ]

    ].copy()

    unique_df.columns = [

        "Full Name",

        "Sex",

        "Age"

    ]

    return unique_df


# ============================================================
# NEWLY REGISTERED MEMBERS
# ============================================================

def get_new_members(df):

    if "name" not in df.columns:

        return pd.DataFrame()

    new_members = df[

        ~df["name"].duplicated(

            keep=False

        )

    ].copy()

    new_members = new_members[

        [

            "name",

            "sex",

            "age"

        ]

    ].copy()

    new_members.columns = [

        "Full Name",

        "Sex",

        "Age"

    ]

    return new_members


# ============================================================
# KPI CARDS
# ============================================================

def create_kpi_cards(df):

    total = len(df)

    if (

        not df.empty

        and "sex" in df.columns

    ):

        gender = normalize_gender(

            df["sex"]

        )

        male = (

            gender == "M"

        ).sum()

        female = (

            gender == "F"

        ).sum()

    else:

        male = 0

        female = 0

    return HTML(

        f"""

        <div class="kpi-container">

            <div class="kpi-card">

                <h2>{female}</h2>

                <p>Female</p>

            </div>

            <div class="kpi-card">

                <h2>{male}</h2>

                <p>Male</p>

            </div>

            <div class="kpi-card">

                <h2>{total}</h2>

                <p>Total Participants</p>

            </div>

        </div>

        """

    )


# ============================================================
# MAIN SECTION
# ============================================================

main_section = widgets.ToggleButtons(

    options=[

        "Onboarding",

        "Engagement Graph",

        "Other Report"

    ],

    value="Onboarding",

    layout=widgets.Layout(

        width="100%"

    )

)


# ============================================================
# ONBOARDING CONTROLS
# ============================================================

onboarding_option = widgets.Dropdown(

    options=[

        "Total Participants",

        "Duplicated Participants",

        "Unique Participants",

        "Newly Registered Members"

    ],

    value="Total Participants",

    description="Option:",

    layout=widgets.Layout(

        width="450px"

    )

)


view_list_button = widgets.Button(

    description="📋 View List",

    button_style="primary",

    layout=widgets.Layout(

        width="180px"

    )

)


onboarding_output = widgets.Output()


# ============================================================
# ENGAGEMENT CONTROLS
# ============================================================

engagement_mode = widgets.Dropdown(

    options=[

        "Age Distribution",

        "Gender",

        "Report Type",

        "Activity / Skills"

    ],

    value="Age Distribution",

    description="Analysis:",

    layout=widgets.Layout(

        width="450px"

    )

)


engagement_gender = widgets.Dropdown(

    options=[

        "Male",

        "Female",

        "Both"

    ],

    value="Both",

    description="Gender:",

    layout=widgets.Layout(

        width="400px"

    )

)


engagement_graph_type = widgets.Dropdown(

    options=[

        "Histogram",

        "Line Graph"

    ],

    value="Histogram",

    description="Graph Type:",

    layout=widgets.Layout(

        width="400px"

    )

)


engagement_activity = widgets.Dropdown(

    options=[

        "Online Course",

        "Internet",

        "Skills Practice",

        "Face to Face Training",

        "E-Library",

        "All Activity / Skills"

    ],

    value="All Activity / Skills",

    description="Activity:",

    layout=widgets.Layout(

        width="550px"

    )

)


engagement_graph_button = widgets.Button(

    description="📊 Generate Graph",

    button_style="primary",

    layout=widgets.Layout(

        width="180px"

    )

)


engagement_output = widgets.Output()


# ============================================================
# OTHER REPORT CONTROLS
# ============================================================

other_gender = widgets.Dropdown(

    options=[

        "Male",

        "Female",

        "Both"

    ],

    value="Both",

    description="Gender:",

    layout=widgets.Layout(

        width="400px"

    )

)


other_activity = widgets.Dropdown(

    options=[

        "Online Course",

        "Internet",

        "Skills Practice",

        "Face to Face Training",

        "E-Library",

        "All Activity / Skills"

    ],

    value="All Activity / Skills",

    description="Activity:",

    layout=widgets.Layout(

        width="550px"

    )

)


search_by = widgets.Dropdown(

    options=[

        "Full Name",

        "Age",

        "Day / Date",

        "Number of Activities Taken"

    ],

    value="Full Name",

    description="Search By:",

    layout=widgets.Layout(

        width="500px"

    )

)


search_box = widgets.Text(

    placeholder="Enter search value...",

    description="Search:",

    layout=widgets.Layout(

        width="500px"

    )

)


other_report_output = widgets.Output()


# ============================================================
# ONBOARDING DISPLAY
# ============================================================

def update_onboarding(change=None):

    with onboarding_output:

        clear_output(
            wait=True
        )

        valid, message = (
            validate_selection()
        )

        if not valid:

            print(message)

            return

        df = (
            get_current_dataframe()
        )

        if df.empty:

            print(
                "❌ No selected data found."
            )

            return

        option = (
            onboarding_option.value
        )

        # TOTAL PARTICIPANTS
        if option == "Total Participants":

            display(
                create_kpi_cards(df)
            )

            print(
                "Total Participants:",
                len(df)
            )

        # DUPLICATED PARTICIPANTS
        elif option == "Duplicated Participants":

            duplicate_summary = (

                get_duplicate_summary(df)

            )

            if duplicate_summary.empty:

                print(
                    "✅ No duplicated participants found."
                )

                return

            duplicate_names = (

                duplicate_summary[
                    "Full Name"
                ]

                .tolist()

            )

            duplicate_df = df[

                df["name"].isin(

                    duplicate_names

                )

            ]

            display(

                create_kpi_cards(

                    duplicate_df

                )

            )

            print(

                "Duplicated Participants:",

                len(duplicate_summary)

            )

        # UNIQUE PARTICIPANTS
        elif option == "Unique Participants":

            unique_df = (

                get_unique_participants(

                    df

                )

            )

            unique_df_for_kpi = (

                unique_df

                .rename(

                    columns={

                        "Full Name": "name",

                        "Sex": "sex",

                        "Age": "age"

                    }

                )

            )

            display(

                create_kpi_cards(

                    unique_df_for_kpi

                )

            )

            print(

                "Unique Participants:",

                len(unique_df)

            )

        # NEW MEMBERS
        elif option == "Newly Registered Members":

            new_members = (

                get_new_members(

                    df

                )

            )

            new_members_for_kpi = (

                new_members

                .rename(

                    columns={

                        "Full Name": "name",

                        "Sex": "sex",

                        "Age": "age"

                    }

                )

            )

            display(

                create_kpi_cards(

                    new_members_for_kpi

                )

            )

            print(

                "Newly Registered Members:",

                len(new_members)

            )


# ============================================================
# ONBOARDING VIEW LIST
# ============================================================

def view_onboarding_list(button=None):

    with onboarding_output:

        clear_output(
            wait=True
        )

        valid, message = (
            validate_selection()
        )

        if not valid:

            print(message)

            return

        df = (
            get_current_dataframe()
        )

        option = (
            onboarding_option.value
        )

        # TOTAL
        if option == "Total Participants":

            table = df[

                [

                    "name",

                    "sex",

                    "age"

                ]

            ].copy()

            table.columns = [

                "Full Name",

                "Sex",

                "Age"

            ]

            table.index = range(

                1,

                len(table) + 1

            )

            display(
                table
            )

        # DUPLICATES
        elif option == "Duplicated Participants":

            duplicate_summary = (

                get_duplicate_summary(

                    df

                )

            )

            duplicate_summary.index = range(

                1,

                len(duplicate_summary) + 1

            )

            display(

                duplicate_summary

            )

        # UNIQUE
        elif option == "Unique Participants":

            unique_df = (

                get_unique_participants(

                    df

                )

            )

            unique_df.index = range(

                1,

                len(unique_df) + 1

            )

            display(
                unique_df
            )

        # NEW MEMBERS
        elif option == "Newly Registered Members":

            new_members = (

                get_new_members(

                    df

                )

            )

            new_members.index = range(

                1,

                len(new_members) + 1

            )

            display(

                new_members

            )


# ============================================================
# ENGAGEMENT GRAPH
# ============================================================

def draw_engagement_graph(button=None):

    with engagement_output:

        clear_output(
            wait=True
        )

        valid, message = (
            validate_selection()
        )

        if not valid:

            print(message)

            return

        df = (
            get_current_dataframe()
        )

        if df.empty:

            print(
                "❌ No selected data found."
            )

            return

        # GENDER FILTER
        selected_gender = (
            engagement_gender.value
        )

        filtered_df = (
            df.copy()
        )

        if selected_gender != "Both":

            gender_code = (

                "M"

                if selected_gender == "Male"

                else "F"

            )

            filtered_df = (

                filtered_df[

                    normalize_gender(

                        filtered_df["sex"]

                    )

                    == gender_code

                ]

            )

        selected_mode = (
            engagement_mode.value
        )

        graph_type = (
            engagement_graph_type.value
        )

        # ====================================================
        # AGE DISTRIBUTION
        # ====================================================

        if selected_mode == "Age Distribution":

            age_df = (
                filtered_df.copy()
            )

            age_df["age"] = pd.to_numeric(

                age_df["age"],

                errors="coerce"

            )

            age_df = age_df.dropna(

                subset=["age"]

            )

            if age_df.empty:

                print(
                    "❌ No valid age data found."
                )

                return

            if graph_type == "Histogram":

                fig = px.histogram(

                    age_df,

                    x="age",

                    nbins=15,

                    title=(

                        "Age Distribution Histogram"

                    )

                )

            else:

                age_counts = (

                    age_df["age"]

                    .value_counts()

                    .sort_index()

                    .reset_index()

                )

                age_counts.columns = [

                    "Age",

                    "Participants"

                ]

                fig = px.line(

                    age_counts,

                    x="Age",

                    y="Participants",

                    markers=True,

                    title=(

                        "Age Distribution Line Graph"

                    )

                )

            fig.show()

        # ====================================================
        # GENDER
        # ====================================================

        elif selected_mode == "Gender":

            gender_df = (

                normalize_gender(

                    filtered_df["sex"]

                )

                .value_counts()

                .reset_index()

            )

            gender_df.columns = [

                "Gender",

                "Count"

            ]

            gender_df["Gender"] = (

                gender_df["Gender"]

                .replace(

                    {

                        "M": "Male",

                        "F": "Female"

                    }

                )

            )

            if graph_type == "Histogram":

                fig = px.histogram(

                    gender_df,

                    x="Gender",

                    y="Count",

                    text_auto=True,

                    title=(

                        "Gender Distribution"

                    )

                )

            else:

                fig = px.line(

                    gender_df,

                    x="Gender",

                    y="Count",

                    markers=True,

                    text="Count",

                    title=(

                        "Gender Distribution"

                    )

                )

            fig.show()

        # ====================================================
        # REPORT TYPE ANALYSIS
        # ====================================================

        elif selected_mode == "Report Type":

            activity_df = (

                get_activity_data(

                    filtered_df

                )

            )

            if activity_df.empty:

                print(

                    "❌ No activity records found."

                )

                return

            report_type = (

                report_type_selector.value

            )

            # DAILY
            if report_type == "Daily":

                report = (

                    activity_df

                    .groupby(

                        [

                            "Source Dataset",

                            "Date"

                        ]

                    )

                    .size()

                    .reset_index(

                        name="Participation"

                    )

                )

                x_column = "Date"

                title = (

                    "Daily Participation"

                )

            # WEEKLY
            elif report_type == "Weekly":

                activity_df["Day_Number"] = (

                    activity_df["Date"]

                    .astype(str)

                    .str.extract(

                        r"(\d+)$"

                    )[0]

                    .astype(float)

                )

                activity_df["Week"] = (

                    (

                        activity_df["Day_Number"]

                        - 1

                    )

                    // 7

                    + 1

                )

                report = (

                    activity_df

                    .groupby(

                        [

                            "Source Dataset",

                            "Week"

                        ]

                    )

                    .size()

                    .reset_index(

                        name="Participation"

                    )

                )

                x_column = "Week"

                title = (

                    "Weekly Participation"

                )

            # MONTHLY
            elif report_type == "Monthly":

                report = (

                    activity_df

                    .groupby(

                        "Source Dataset"

                    )

                    .size()

                    .reset_index(

                        name="Participation"

                    )

                )

                x_column = (
                    "Source Dataset"
                )

                title = (

                    "Monthly Participation"

                )

            # MULTI-MONTH
            elif report_type in [

                "2 Months",

                "3 Months",

                "4 Months",

                "5 Months",

                "6 Months",

                "7 Months",

                "8 Months",

                "9 Months",

                "10 Months",

                "11 Months"

            ]:

                report = (

                    activity_df

                    .groupby(

                        "Source Dataset"

                    )

                    .size()

                    .reset_index(

                        name="Participation"

                    )

                )

                x_column = (

                    "Source Dataset"

                )

                title = (

                    report_type

                    + " Participation"

                )

            # YEARLY
            else:

                report = pd.DataFrame(

                    {

                        "Year": [

                            "Yearly Total"

                        ],

                        "Participation": [

                            len(activity_df)

                        ]

                    }

                )

                x_column = "Year"

                title = (

                    "Yearly Participation"

                )

            if graph_type == "Histogram":

                fig = px.histogram(

                    report,

                    x=x_column,

                    y="Participation",

                    text_auto=True,

                    title=title

                )

            else:

                fig = px.line(

                    report,

                    x=x_column,

                    y="Participation",

                    markers=True,

                    text="Participation",

                    title=title

                )

            fig.show()

        # ====================================================
        # ACTIVITY / SKILLS
        # ====================================================

        elif selected_mode == "Activity / Skills":

            activity_df = (

                get_activity_data(

                    filtered_df

                )

            )

            if activity_df.empty:

                print(

                    "❌ No activity records found."

                )

                return

            selected_activity = (

                engagement_activity.value

            )

            if (

                selected_activity

                == "All Activity / Skills"

            ):

                activity_counts = (

                    activity_df[

                        "Activity"

                    ]

                    .value_counts()

                    .reset_index()

                )

                activity_counts.columns = [

                    "Activity",

                    "Participation"

                ]

                if graph_type == "Histogram":

                    fig = px.histogram(

                        activity_counts,

                        x="Activity",

                        y="Participation",

                        text_auto=True,

                        title=(

                            "All Activity / Skills"

                        )

                    )

                else:

                    fig = px.line(

                        activity_counts,

                        x="Activity",

                        y="Participation",

                        markers=True,

                        text="Participation",

                        title=(

                            "All Activity / Skills"

                        )

                    )

            else:

                matching_activity = (

                    activity_df[

                        activity_df[

                            "Activity"

                        ]

                        .str.contains(

                            selected_activity,

                            case=False,

                            na=False

                        )

                    ]

                )

                if matching_activity.empty:

                    print(

                        "❌ No matching activity found."

                    )

                    return

                activity_counts = (

                    matching_activity

                    .groupby(

                        [

                            "Source Dataset",

                            "Date"

                        ]

                    )

                    .size()

                    .reset_index(

                        name="Participation"

                    )

                )

                if graph_type == "Histogram":

                    fig = px.histogram(

                        activity_counts,

                        x="Date",

                        y="Participation",

                        text_auto=True,

                        title=(

                            selected_activity

                            + " Participation"

                        )

                    )

                else:

                    fig = px.line(

                        activity_counts,

                        x="Date",

                        y="Participation",

                        markers=True,

                        text="Participation",

                        title=(

                            selected_activity

                            + " Participation"

                        )

                    )

            fig.show()


# ============================================================
# OTHER REPORT
# ============================================================

def generate_other_report(change=None):

    with other_report_output:

        clear_output(
            wait=True
        )

        valid, message = (
            validate_selection()
        )

        if not valid:

            print(message)

            return

        df = (
            get_current_dataframe()
        )

        if df.empty:

            print(
                "❌ No selected data found."
            )

            return

        # GENDER FILTER
        selected_gender = (
            other_gender.value
        )

        filtered_df = (
            df.copy()
        )

        if selected_gender != "Both":

            gender_code = (

                "M"

                if selected_gender == "Male"

                else "F"

            )

            filtered_df = (

                filtered_df[

                    normalize_gender(

                        filtered_df["sex"]

                    )

                    == gender_code

                ]

            )

        activity_df = (

            get_activity_data(

                filtered_df

            )

        )

        if activity_df.empty:

            print(

                "❌ No activity records found."

            )

            return

        # ACTIVITY FILTER
        selected_activity = (

            other_activity.value

        )

        if (

            selected_activity

            != "All Activity / Skills"

        ):

            activity_df = (

                activity_df[

                    activity_df[

                        "Activity"

                    ]

                    .str.contains(

                        selected_activity,

                        case=False,

                        na=False

                    )

                ]

            )

        search_value = (

            search_box.value

            .strip()

            .lower()

        )

        search_type = (
            search_by.value
        )

        if search_value:

            # FULL NAME
            if search_type == "Full Name":

                activity_df = (

                    activity_df[

                        activity_df[

                            "Name"

                        ]

                        .astype(str)

                        .str.lower()

                        .str.contains(

                            search_value,

                            na=False

                        )

                    ]

                )

            # AGE
            elif search_type == "Age":

                activity_df = (

                    activity_df[

                        activity_df[

                            "Age"

                        ]

                        .astype(str)

                        .str.contains(

                            search_value,

                            na=False

                        )

                    ]

                )

            # DATE
            elif search_type == "Day / Date":

                activity_df = (

                    activity_df[

                        activity_df[

                            "Date"

                        ]

                        .astype(str)

                        .str.lower()

                        .str.contains(

                            search_value,

                            na=False

                        )

                    ]

                )

            # NUMBER OF ACTIVITIES
            elif (

                search_type

                == "Number of Activities Taken"

            ):

                activity_count = (

                    activity_df

                    .groupby(

                        [

                            "Name",

                            "Sex",

                            "Age"

                        ]

                    )

                    .size()

                    .reset_index(

                        name=(

                            "Number of Activities Taken"

                        )

                    )

                )

                activity_count = (

                    activity_count[

                        activity_count[

                            "Number of Activities Taken"

                        ]

                        .astype(str)

                        .str.contains(

                            search_value,

                            na=False

                        )

                    ]

                )

                activity_count.index = range(

                    1,

                    len(activity_count) + 1

                )

                display(

                    activity_count

                )

                return

        result = activity_df[

            [

                "Name",

                "Sex",

                "Age",

                "Date",

                "Activity",

                "Source Dataset"

            ]

        ].copy()

        result.index = range(

            1,

            len(result) + 1

        )

        display(
            result
        )


# ============================================================
# DISPLAY ONBOARDING SECTION
# ============================================================

def display_onboarding_section():

    display(

        HTML(

            "<h2 class='section-title'>👥 Onboarding</h2>"

        )

    )

    display(

        widgets.HBox(

            [

                onboarding_option,

                view_list_button

            ]

        )

    )

    display(

        onboarding_output

    )

    update_onboarding()


# ============================================================
# DISPLAY ENGAGEMENT SECTION
# ============================================================

def display_engagement_section():

    display(

        HTML(

            "<h2 class='section-title'>📈 Engagement Graph</h2>"

        )

    )

    display(

        widgets.VBox(

            [

                engagement_mode,

                engagement_gender,

                engagement_graph_type,

                engagement_activity,

                engagement_graph_button

            ]

        )

    )

    display(

        engagement_output

    )


# ============================================================
# DISPLAY OTHER REPORT SECTION
# ============================================================

def display_other_report_section():

    display(

        HTML(

            "<h2 class='section-title'>📊 Other Report</h2>"

        )

    )

    display(

        widgets.VBox(

            [

                other_gender,

                other_activity,

                search_by,

                search_box

            ]

        )

    )

    display(

        other_report_output

    )

    generate_other_report()


# ============================================================
# MAIN DASHBOARD DISPLAY
# ============================================================

def display_dashboard(change=None):

    clear_output(
        wait=True
    )

    display(

        HTML(

            "<h1 class='dashboard-title'>"

            "📊 Attendance Dashboard"

            "</h1>"

        )

    )

    # --------------------------------------------------------
    # GLOBAL DATASET SELECTION
    # --------------------------------------------------------

    display(

        HTML(

            "<h3>📁 Select Monthly Dataset(s)</h3>"

        )

    )

    display(
        month_selector
    )

    # --------------------------------------------------------
    # GLOBAL REPORT TYPE
    # --------------------------------------------------------

    display(

        HTML(

            "<h3>📅 Select Report Type</h3>"

        )

    )

    display(
        report_type_selector
    )

    display(
        global_status_output
    )

    update_global_status()

    # --------------------------------------------------------
    # MAIN SECTION BUTTONS
    # --------------------------------------------------------

    display(
        main_section
    )

    if (

        main_section.value

        == "Onboarding"

    ):

        display_onboarding_section()

    elif (

        main_section.value

        == "Engagement Graph"

    ):

        display_engagement_section()

    elif (

        main_section.value

        == "Other Report"

    ):

        display_other_report_section()


# ============================================================
# EVENT CONNECTIONS
# ============================================================

# Dataset selection
month_selector.observe(

    update_global_status,

    names="value"

)

month_selector.observe(

    display_dashboard,

    names="value"

)


# Report type selection
report_type_selector.observe(

    update_global_status,

    names="value"

)

report_type_selector.observe(

    display_dashboard,

    names="value"

)


# Main section
main_section.observe(

    display_dashboard,

    names="value"

)


# Onboarding
onboarding_option.observe(

    update_onboarding,

    names="value"

)

view_list_button.on_click(

    view_onboarding_list

)


# Engagement
engagement_graph_button.on_click(

    draw_engagement_graph

)


# Other report
other_gender.observe(

    generate_other_report,

    names="value"

)

other_activity.observe(

    generate_other_report,

    names="value"

)

search_by.observe(

    generate_other_report,

    names="value"

)

search_box.observe(

    generate_other_report,

    names="value"

)


# ============================================================
# START DASHBOARD
# ============================================================

display_dashboard()

SelectMultiple(description='Month Dataset:', index=(0,), layout=Layout(width='650px'), options=('DIY Attendanc…

Dropdown(description='Report Type:', index=2, layout=Layout(width='450px'), options=('Daily', 'Weekly', 'Month…

Output()

ToggleButtons(layout=Layout(width='100%'), options=('Onboarding', 'Engagement Graph', 'Other Report'), value='…

Output()

In [20]:
# ============================================================
# COMPLETE INTERACTIVE ATTENDANCE DASHBOARD
# GLOBAL REPORT TYPE + MONTH CHECKBOX DROPDOWN
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, clear_output, HTML

import plotly.express as px


# ============================================================
# BASIC DATA CHECK
# ============================================================

if not datasets:

    print("❌ No datasets found.")

else:

    print(
        f"✅ {len(datasets)} dataset(s) available for dashboard."
    )


# ============================================================
# MONTHS
# ============================================================

MONTHS = [

    "January",

    "February",

    "March",

    "April",

    "May",

    "June",

    "July",

    "August",

    "September",

    "October",

    "November",

    "December"

]


# ============================================================
# DASHBOARD CSS
# ============================================================

display(

    HTML(

        """

        <style>

        .dashboard-title {

            font-size: 30px;

            font-weight: bold;

            margin-bottom: 20px;

        }


        .section-title {

            font-size: 23px;

            font-weight: bold;

            margin-top: 20px;

            margin-bottom: 15px;

        }


        .kpi-container {

            display: flex;

            gap: 18px;

            margin: 20px 0;

            flex-wrap: wrap;

        }


        .kpi-card {

            min-width: 180px;

            padding: 20px;

            border-radius: 12px;

            text-align: center;

            box-shadow: 0px 3px 10px rgba(0,0,0,0.15);

            background-color: #f8f9fa;

        }


        .kpi-card h2 {

            font-size: 32px;

            margin: 0;

        }


        .kpi-card p {

            font-size: 16px;

            margin-top: 8px;

        }


        .report-box {

            border: 1px solid #cccccc;

            padding: 15px;

            border-radius: 10px;

            margin-bottom: 15px;

        }

        </style>

        """

    )

)


# ============================================================
# DATASET SELECTOR
# ============================================================

file_selector = widgets.Dropdown(

    options=list(datasets.keys()),

    description="Dataset:",

    layout=widgets.Layout(

        width="500px"

    )

)


# ============================================================
# REPORT TYPE
# ============================================================

report_type = widgets.Dropdown(

    options=[

        "Daily",

        "Weekly",

        "Monthly",

        "Other",

        "Yearly"

    ],

    value="Daily",

    description="Report Type:",

    layout=widgets.Layout(

        width="500px"

    )

)


# ============================================================
# GET MONTH FROM DATASET NAME
# ============================================================

def get_month_from_filename(filename):

    filename_text = str(filename).lower()


    for month in MONTHS:

        if month.lower() in filename_text:

            return month


    return None


# ============================================================
# GET UPLOADED MONTHS
# ============================================================

def get_uploaded_months():

    uploaded_months = []


    for filename in datasets.keys():

        # First try dataset_months if already available

        if "dataset_months" in globals():

            if filename in dataset_months:

                month = dataset_months[filename]

                if month in MONTHS:

                    uploaded_months.append(month)

                    continue


        # Otherwise detect month from filename

        month = get_month_from_filename(filename)


        if month is not None:

            uploaded_months.append(month)


    return sorted(

        list(set(uploaded_months)),

        key=lambda x: MONTHS.index(x)

    )


# ============================================================
# MONTH CHECKBOXES
# ============================================================

month_checkboxes = {

    month: widgets.Checkbox(

        value=False,

        description=month,

        disabled=True,

        indent=False,

        layout=widgets.Layout(

            width="180px"

        )

    )

    for month in MONTHS

}


# ============================================================
# MONTH DROPDOWN TOGGLE
# ============================================================

month_selector_button = widgets.ToggleButton(

    value=False,

    description="📅 Select Month(s)",

    layout=widgets.Layout(

        width="250px"

    )

)


month_checkbox_box = widgets.VBox(

    list(

        month_checkboxes.values()

    ),

    layout=widgets.Layout(

        border="1px solid #cccccc",

        padding="10px",

        width="250px",

        max_height="260px",

        overflow_y="auto"

    )

)


month_checkbox_box.layout.display = "none"


month_dropdown = widgets.VBox(

    [

        month_selector_button,

        month_checkbox_box

    ]

)


# ============================================================
# TOGGLE MONTH LIST
# ============================================================

def toggle_month_dropdown(change):

    if change["new"]:

        month_checkbox_box.layout.display = "flex"

        month_selector_button.description = (

            "📅 Hide Month(s)"

        )

    else:

        month_checkbox_box.layout.display = "none"

        month_selector_button.description = (

            "📅 Select Month(s)"

        )


month_selector_button.observe(

    toggle_month_dropdown,

    names="value"

)


# ============================================================
# GET SELECTED MONTHS
# ============================================================

def get_selected_months():

    return [

        month

        for month in MONTHS

        if month_checkboxes[month].value

    ]


# ============================================================
# UPDATE MONTH CHECKBOXES
# ============================================================

def update_month_checkboxes():

    uploaded_months = get_uploaded_months()


    for month in MONTHS:

        checkbox = month_checkboxes[month]


        if month in uploaded_months:

            checkbox.disabled = False

        else:

            checkbox.disabled = True

            checkbox.value = False


    # YEARLY ONLY IF ALL 12 MONTHS EXIST

    if len(uploaded_months) == 12:

        report_type.options = [

            "Daily",

            "Weekly",

            "Monthly",

            "Other",

            "Yearly"

        ]

    else:

        report_type.options = [

            "Daily",

            "Weekly",

            "Monthly",

            "Other"

        ]


        if report_type.value == "Yearly":

            report_type.value = "Daily"


# ============================================================
# GET DATASET BY MONTH
# ============================================================

def get_dataset_for_month(month):

    uploaded_months = get_uploaded_months()


    for filename in datasets.keys():

        detected_month = None


        if "dataset_months" in globals():

            if filename in dataset_months:

                detected_month = dataset_months[filename]


        if detected_month is None:

            detected_month = get_month_from_filename(filename)


        if detected_month == month:

            return datasets[filename].copy()


    return pd.DataFrame()


# ============================================================
# COMBINE SELECTED MONTHS
# ============================================================

def get_selected_period_dataframe():

    selected_months = get_selected_months()


    if not selected_months:

        return pd.DataFrame()


    frames = []


    for month in selected_months:

        month_df = get_dataset_for_month(month)


        if not month_df.empty:

            month_df = month_df.copy()


            month_df["Report_Month"] = month


            frames.append(month_df)


    if not frames:

        return pd.DataFrame()


    return pd.concat(

        frames,

        ignore_index=True

    )


# ============================================================
# NORMALIZE GENDER
# ============================================================

def normalize_gender(series):

    return (

        series

        .astype(str)

        .str.strip()

        .str.upper()

        .replace(

            {

                "MALE": "M",

                "FEMALE": "F"

            }

        )

    )


# ============================================================
# GET ACTIVITY COLUMNS
# ============================================================

def get_activity_columns(df):

    activity_columns = []


    for column in df.columns:

        column_text = str(column).strip().lower()


        if column_text in [

            "name",

            "sex",

            "age",

            "report_month"

        ]:

            continue


        if column_text.isdigit():

            day_number = int(column_text)


            if 1 <= day_number <= 31:

                activity_columns.append(column)

                continue


        if any(

            month.lower() in column_text

            for month in MONTHS

        ):

            activity_columns.append(column)


    return activity_columns


# ============================================================
# CONVERT WIDE DATA TO LONG ACTIVITY DATA
# ============================================================

def get_activity_data(df):

    activity_columns = get_activity_columns(df)


    if not activity_columns:

        return pd.DataFrame(

            columns=[

                "Name",

                "Sex",

                "Age",

                "Date",

                "Activity",

                "Report_Month"

            ]

        )


    records = []


    for column in activity_columns:


        selected_columns = [

            "name",

            "sex",

            "age",

            column

        ]


        if "Report_Month" in df.columns:

            selected_columns.append(

                "Report_Month"

            )


        temp = df[

            selected_columns

        ].copy()


        rename_dict = {

            "name": "Name",

            "sex": "Sex",

            "age": "Age",

            column: "Activity"

        }


        temp = temp.rename(

            columns=rename_dict

        )


        temp["Date"] = str(column)


        records.append(temp)


    activity_df = pd.concat(

        records,

        ignore_index=True

    )


    activity_df["Activity"] = (

        activity_df["Activity"]

        .astype(str)

        .str.strip()

    )


    invalid_values = [

        "",

        "0",

        "0.0",

        "nan",

        "NaN",

        "None",

        "N/A",

        "NA"

    ]


    activity_df = activity_df[

        ~activity_df["Activity"].isin(

            invalid_values

        )

    ].copy()


    activity_df["Age"] = pd.to_numeric(

        activity_df["Age"],

        errors="coerce"

    )


    return activity_df


# ============================================================
# DUPLICATE SUMMARY
# ============================================================

def get_duplicate_summary(df):

    if "name" not in df.columns:

        return pd.DataFrame()


    duplicate_rows = df[

        df.duplicated(

            subset=["name"],

            keep=False

        )

    ].copy()


    if duplicate_rows.empty:

        return pd.DataFrame(

            columns=[

                "Full Name",

                "Sex",

                "Age",

                "Duplication_Count",

                "Duplicated_Index"

            ]

        )


    duplicate_summary = (

        duplicate_rows

        .groupby("name")

        .agg(

            Sex=(

                "sex",

                lambda x: list(x)

            ),

            Age=(

                "age",

                lambda x: list(x)

            ),

            Duplication_Count=(

                "name",

                "count"

            ),

            Duplicated_Index=(

                "name",

                lambda x: list(x.index)

            )

        )

        .reset_index()

    )


    duplicate_summary = (

        duplicate_summary

        .rename(

            columns={

                "name": "Full Name"

            }

        )

    )


    return duplicate_summary[

        [

            "Full Name",

            "Sex",

            "Age",

            "Duplication_Count",

            "Duplicated_Index"

        ]

    ]


# ============================================================
# NEW MEMBERS
# ============================================================

def get_new_members(df):

    if "name" not in df.columns:

        return pd.DataFrame()


    new_members = df[

        ~df["name"].duplicated(

            keep=False

        )

    ].copy()


    new_members = new_members[

        [

            "name",

            "sex",

            "age"

        ]

    ].copy()


    new_members.columns = [

        "Full Name",

        "Sex",

        "Age"

    ]


    return new_members


# ============================================================
# KPI CARDS
# ============================================================

def create_kpi_cards(df):

    total = len(df)


    if (

        not df.empty

        and "sex" in df.columns

    ):

        gender = normalize_gender(

            df["sex"]

        )


        male = (

            gender == "M"

        ).sum()


        female = (

            gender == "F"

        ).sum()

    else:

        male = 0

        female = 0


    return HTML(

        f"""

        <div class="kpi-container">


            <div class="kpi-card">

                <h2>{female}</h2>

                <p>Female</p>

            </div>


            <div class="kpi-card">

                <h2>{male}</h2>

                <p>Male</p>

            </div>


            <div class="kpi-card">

                <h2>{total}</h2>

                <p>Total Participants</p>

            </div>


        </div>

        """

    )


# ============================================================
# GLOBAL REPORT CONTROLS
# ============================================================

daily_day = widgets.Dropdown(

    options=["All Days"] + list(range(1, 32)),

    value="All Days",

    description="Day:",

    layout=widgets.Layout(

        width="350px"

    )

)


weekly_mode = widgets.Dropdown(

    options=[

        "Automatic 7-Day Intervals",

        "Custom Start and End Day"

    ],

    value="Automatic 7-Day Intervals",

    description="Weekly Mode:",

    layout=widgets.Layout(

        width="500px"

    )

)


weekly_start_day = widgets.IntText(

    value=1,

    min=1,

    max=31,

    description="Start Day:",

    layout=widgets.Layout(

        width="300px"

    )

)


weekly_end_day = widgets.IntText(

    value=7,

    min=1,

    max=31,

    description="End Day:",

    layout=widgets.Layout(

        width="300px"

    )

)


# ============================================================
# MAIN SECTION
# ============================================================

main_section = widgets.ToggleButtons(

    options=[

        "Onboarding",

        "Engagement Graph",

        "Other Report"

    ],

    value="Onboarding",

    layout=widgets.Layout(

        width="100%"

    )

)


# ============================================================
# ONBOARDING CONTROLS
# ============================================================

onboarding_option = widgets.Dropdown(

    options=[

        "Total Participants",

        "Duplicated Participants",

        "Newly Registered Members"

    ],

    value="Total Participants",

    description="Option:",

    layout=widgets.Layout(

        width="500px"

    )

)


view_list_button = widgets.Button(

    description="📋 View List",

    button_style="primary",

    layout=widgets.Layout(

        width="180px"

    )

)


onboarding_output = widgets.Output()


# ============================================================
# ENGAGEMENT CONTROLS
# ============================================================

engagement_mode = widgets.Dropdown(

    options=[

        "Gender",

        "Age Distribution",

        "Activity / Skills"

    ],

    value="Gender",

    description="Analysis:",

    layout=widgets.Layout(

        width="500px"

    )

)


engagement_gender = widgets.Dropdown(

    options=[

        "Male",

        "Female",

        "Both"

    ],

    value="Both",

    description="Gender:",

    layout=widgets.Layout(

        width="400px"

    )

)


engagement_graph_type = widgets.Dropdown(

    options=[

        "Histogram",

        "Line Graph"

    ],

    value="Histogram",

    description="Graph Type:",

    layout=widgets.Layout(

        width="400px"

    )

)


engagement_activity = widgets.Dropdown(

    options=[

        "Online Course",

        "Internet",

        "Skills Practice",

        "Face to Face Training",

        "E-Library",

        "All Activity / Skills"

    ],

    value="All Activity / Skills",

    description="Activity:",

    layout=widgets.Layout(

        width="550px"

    )

)


engagement_graph_button = widgets.Button(

    description="📊 Graph",

    button_style="primary",

    layout=widgets.Layout(

        width="180px"

    )

)


engagement_output = widgets.Output()


# ============================================================
# OTHER REPORT CONTROLS
# ============================================================

other_gender = widgets.Dropdown(

    options=[

        "Male",

        "Female",

        "Both"

    ],

    value="Both",

    description="Gender:",

    layout=widgets.Layout(

        width="400px"

    )

)


other_activity = widgets.Dropdown(

    options=[

        "Online Course",

        "Internet",

        "Skills Practice",

        "Face to Face Training",

        "E-Library",

        "All Activity / Skills"

    ],

    value="All Activity / Skills",

    description="Activity:",

    layout=widgets.Layout(

        width="550px"

    )

)


search_by = widgets.Dropdown(

    options=[

        "Full Name",

        "Age",

        "Day / Date",

        "Number of Activities Taken"

    ],

    value="Full Name",

    description="Search By:",

    layout=widgets.Layout(

        width="500px"

    )

)


search_box = widgets.Text(

    placeholder="Enter search value...",

    description="Search:",

    layout=widgets.Layout(

        width="500px"

    )

)


other_report_output = widgets.Output()


# ============================================================
# REPORT VALIDATION
# ============================================================

def validate_report_selection():

    selected_months = get_selected_months()

    selected_report = report_type.value


    if selected_report == "Yearly":

        if len(get_uploaded_months()) != 12:

            return False, (

                "❌ Yearly report requires all 12 "

                "monthly datasets."

            )


        return True, ""


    if selected_report in [

        "Daily",

        "Weekly",

        "Monthly"

    ]:

        if len(selected_months) != 1:

            return False, (

                f"❌ {selected_report} report requires "

                "exactly ONE month."

            )


    if selected_report == "Other":

        if len(selected_months) < 1:

            return False, (

                "❌ Other report requires at least "

                "ONE selected month."

            )


    if selected_report == "Weekly":

        if weekly_mode.value == (

            "Custom Start and End Day"

        ):

            start = weekly_start_day.value

            end = weekly_end_day.value


            if start < 1 or end > 31:

                return False, (

                    "❌ Day must be between 1 and 31."

                )


            if end < start:

                return False, (

                    "❌ End day must be after "

                    "start day."

                )


            if end - start + 1 > 7:

                return False, (

                    "❌ Weekly interval cannot "

                    "exceed 7 days."

                )


    return True, ""


# ============================================================
# GET REPORT DATA
# ============================================================

def get_report_data():

    valid, message = validate_report_selection()


    if not valid:

        return pd.DataFrame(), message


    selected_months = get_selected_months()


    if report_type.value == "Yearly":

        selected_months = MONTHS


    frames = []


    for month in selected_months:

        month_df = get_dataset_for_month(month)


        if not month_df.empty:

            month_df = month_df.copy()


            month_df["Report_Month"] = month


            frames.append(month_df)


    if not frames:

        return pd.DataFrame(), (

            "❌ No data found for selected month(s)."

        )


    df = pd.concat(

        frames,

        ignore_index=True

    )


    return df, ""


# ============================================================
# FILTER ACTIVITY DATA BY REPORT TYPE
# ============================================================

def apply_report_filter(activity_df):

    if activity_df.empty:

        return activity_df


    selected_report = report_type.value


    # DAILY

    if selected_report == "Daily":

        selected_day = daily_day.value


        if selected_day != "All Days":

            activity_df = activity_df[

                activity_df["Date"].astype(str)

                .str.extract(r"(\d+)")[0]

                .astype(float)

                == int(selected_day)

            ]


    # WEEKLY

    elif selected_report == "Weekly":

        activity_df = activity_df.copy()


        activity_df["Day_Number"] = (

            activity_df["Date"]

            .astype(str)

            .str.extract(r"(\d+)")[0]

            .astype(float)

        )


        if weekly_mode.value == (

            "Automatic 7-Day Intervals"

        ):

            activity_df["Week"] = (

                (

                    activity_df["Day_Number"]

                    - 1

                )

                // 7

                + 1

            )


        else:

            start = weekly_start_day.value

            end = weekly_end_day.value


            activity_df = activity_df[

                (

                    activity_df["Day_Number"]

                    >= start

                )

                &

                (

                    activity_df["Day_Number"]

                    <= end

                )

            ]


    return activity_df


# ============================================================
# ONBOARDING DISPLAY
# ============================================================

def update_onboarding(change=None):

    with onboarding_output:

        clear_output(wait=True)


        df, message = get_report_data()


        if df.empty:

            print(message)

            return


        option = onboarding_option.value


        # TOTAL PARTICIPANTS

        if option == "Total Participants":

            display(

                create_kpi_cards(df)

            )


            print(

                "Selected Report Type:",

                report_type.value

            )


            print(

                "Selected Month(s):",

                ", ".join(

                    get_selected_months()

                )

                if report_type.value != "Yearly"

                else "All 12 Months"

            )


            print(

                f"Total Participants: {len(df)}"

            )


        # DUPLICATES

        elif option == "Duplicated Participants":

            duplicate_summary = (

                get_duplicate_summary(df)

            )


            duplicate_names = (

                duplicate_summary[

                    "Full Name"

                ]

                .tolist()

            )


            duplicate_df = df[

                df["name"].isin(

                    duplicate_names

                )

            ]


            display(

                create_kpi_cards(

                    duplicate_df

                )

            )


            print(

                "Duplicated Participants:",

                len(duplicate_summary)

            )


        # NEW MEMBERS

        elif option == "Newly Registered Members":

            new_members = (

                get_new_members(df)

            )


            new_members_for_kpi = (

                new_members

                .rename(

                    columns={

                        "Full Name": "name",

                        "Sex": "sex",

                        "Age": "age"

                    }

                )

            )


            display(

                create_kpi_cards(

                    new_members_for_kpi

                )

            )


            print(

                "Newly Registered Members:",

                len(new_members)

            )


# ============================================================
# ONBOARDING VIEW LIST
# ============================================================

def view_onboarding_list(button=None):

    with onboarding_output:

        clear_output(wait=True)


        df, message = get_report_data()


        if df.empty:

            print(message)

            return


        option = onboarding_option.value


        if option == "Total Participants":

            table = df[

                [

                    "name",

                    "sex",

                    "age"

                ]

            ].copy()


            table.columns = [

                "Full Name",

                "Sex",

                "Age"

            ]


            table.index = range(

                1,

                len(table) + 1

            )


            display(table)


        elif option == "Duplicated Participants":

            duplicate_summary = (

                get_duplicate_summary(df)

            )


            duplicate_summary.index = range(

                1,

                len(duplicate_summary) + 1

            )


            display(

                duplicate_summary

            )


        elif option == "Newly Registered Members":

            new_members = (

                get_new_members(df)

            )


            new_members.index = range(

                1,

                len(new_members) + 1

            )


            display(new_members)


# ============================================================
# ENGAGEMENT GRAPH
# ============================================================

def draw_engagement_graph(button=None):

    with engagement_output:

        clear_output(wait=True)


        df, message = get_report_data()


        if df.empty:

            print(message)

            return


        selected_gender = (

            engagement_gender.value

        )


        if selected_gender != "Both":

            gender_code = (

                "M"

                if selected_gender == "Male"

                else "F"

            )


            df = df[

                normalize_gender(

                    df["sex"]

                )

                == gender_code

            ]


        selected_mode = (

            engagement_mode.value

        )


        graph_type = (

            engagement_graph_type.value

        )


        # ====================================================
        # GENDER
        # ====================================================

        if selected_mode == "Gender":


            gender_df = (

                normalize_gender(

                    df["sex"]

                )

                .value_counts()

                .reset_index()

            )


            gender_df.columns = [

                "Gender",

                "Count"

            ]


            gender_df["Gender"] = (

                gender_df["Gender"]

                .replace(

                    {

                        "M": "Male",

                        "F": "Female"

                    }

                )

            )


            if graph_type == "Histogram":

                fig = px.histogram(

                    gender_df,

                    x="Gender",

                    y="Count",

                    text_auto=True,

                    title=(

                        "Gender Distribution - "

                        + report_type.value

                    )

                )

            else:

                fig = px.line(

                    gender_df,

                    x="Gender",

                    y="Count",

                    markers=True,

                    text="Count",

                    title=(

                        "Gender Distribution - "

                        + report_type.value

                    )

                )


            fig.show()


        # ====================================================
        # AGE
        # ====================================================

        elif selected_mode == "Age Distribution":


            age_df = df.copy()


            age_df["age"] = pd.to_numeric(

                age_df["age"],

                errors="coerce"

            )


            age_df = age_df.dropna(

                subset=["age"]

            )


            if age_df.empty:

                print(

                    "❌ No valid age data found."

                )

                return


            if graph_type == "Histogram":

                fig = px.histogram(

                    age_df,

                    x="age",

                    nbins=15,

                    title=(

                        "Age Distribution - "

                        + report_type.value

                    )

                )


            else:

                age_counts = (

                    age_df["age"]

                    .value_counts()

                    .sort_index()

                    .reset_index()

                )


                age_counts.columns = [

                    "Age",

                    "Participants"

                ]


                fig = px.line(

                    age_counts,

                    x="Age",

                    y="Participants",

                    markers=True,

                    title=(

                        "Age Distribution - "

                        + report_type.value

                    )

                )


            fig.show()


        # ====================================================
        # ACTIVITY / SKILLS
        # ====================================================

        elif selected_mode == "Activity / Skills":


            activity_df = (

                get_activity_data(df)

            )


            activity_df = (

                apply_report_filter(

                    activity_df

                )

            )


            if activity_df.empty:

                print(

                    "❌ No activity records found."

                )

                return


            selected_activity = (

                engagement_activity.value

            )


            if (

                selected_activity

                == "All Activity / Skills"

            ):


                activity_counts = (

                    activity_df[

                        "Activity"

                    ]

                    .value_counts()

                    .reset_index()

                )


                activity_counts.columns = [

                    "Activity",

                    "Participation"

                ]


                if graph_type == "Histogram":

                    fig = px.histogram(

                        activity_counts,

                        x="Activity",

                        y="Participation",

                        text_auto=True,

                        title=(

                            "All Activity / Skills"

                        )

                    )

                else:

                    fig = px.line(

                        activity_counts,

                        x="Activity",

                        y="Participation",

                        markers=True,

                        text="Participation",

                        title=(

                            "All Activity / Skills"

                        )

                    )


            else:


                matching_activity = (

                    activity_df[

                        activity_df[

                            "Activity"

                        ]

                        .str.contains(

                            selected_activity,

                            case=False,

                            na=False

                        )

                    ]

                )


                if matching_activity.empty:

                    print(

                        "❌ No matching activity found."

                    )

                    return


                activity_counts = (

                    matching_activity

                    .groupby("Date")

                    .size()

                    .reset_index(

                        name="Participation"

                    )

                )


                if graph_type == "Histogram":

                    fig = px.histogram(

                        activity_counts,

                        x="Date",

                        y="Participation",

                        text_auto=True,

                        title=(

                            selected_activity

                            + " Participation"

                        )

                    )

                else:

                    fig = px.line(

                        activity_counts,

                        x="Date",

                        y="Participation",

                        markers=True,

                        text="Participation",

                        title=(

                            selected_activity

                            + " Participation"

                        )

                    )


            fig.show()


# ============================================================
# OTHER REPORT
# ============================================================

def generate_other_report(change=None):

    with other_report_output:

        clear_output(wait=True)


        df, message = get_report_data()


        if df.empty:

            print(message)

            return


        selected_gender = (

            other_gender.value

        )


        if selected_gender != "Both":

            gender_code = (

                "M"

                if selected_gender == "Male"

                else "F"

            )


            df = df[

                normalize_gender(

                    df["sex"]

                )

                == gender_code

            ]


        activity_df = (

            get_activity_data(df)

        )


        activity_df = (

            apply_report_filter(

                activity_df

            )

        )


        if activity_df.empty:

            print(

                "❌ No activity records found."

            )

            return


        selected_activity = (

            other_activity.value

        )


        if (

            selected_activity

            != "All Activity / Skills"

        ):


            activity_df = activity_df[

                activity_df[

                    "Activity"

                ]

                .str.contains(

                    selected_activity,

                    case=False,

                    na=False

                )

            ]


        search_value = (

            search_box.value

            .strip()

            .lower()

        )


        search_type = (

            search_by.value

        )


        if search_value:


            # FULL NAME

            if search_type == "Full Name":

                activity_df = activity_df[

                    activity_df[

                        "Name"

                    ]

                    .astype(str)

                    .str.lower()

                    .str.contains(

                        search_value,

                        na=False

                    )

                ]


            # AGE

            elif search_type == "Age":

                activity_df = activity_df[

                    activity_df[

                        "Age"

                    ]

                    .astype(str)

                    .str.contains(

                        search_value,

                        na=False

                    )

                ]


            # DAY / DATE

            elif search_type == "Day / Date":

                activity_df = activity_df[

                    activity_df[

                        "Date"

                    ]

                    .astype(str)

                    .str.lower()

                    .str.contains(

                        search_value,

                        na=False

                    )

                ]


            # NUMBER OF ACTIVITIES

            elif (

                search_type

                == "Number of Activities Taken"

            ):


                activity_count = (

                    activity_df

                    .groupby(

                        [

                            "Name",

                            "Sex",

                            "Age"

                        ]

                    )

                    .size()

                    .reset_index(

                        name=(

                            "Number of Activities Taken"

                        )

                    )

                )


                activity_count = (

                    activity_count[

                        activity_count[

                            "Number of Activities Taken"

                        ]

                        .astype(str)

                        .str.contains(

                            search_value,

                            na=False

                        )

                    ]

                )


                activity_count.index = range(

                    1,

                    len(activity_count) + 1

                )


                display(

                    activity_count

                )


                return


        result = activity_df[

            [

                "Name",

                "Sex",

                "Age",

                "Date",

                "Activity"

            ]

        ].copy()


        result.index = range(

            1,

            len(result) + 1

        )


        display(result)


# ============================================================
# DISPLAY REPORT CONTROLS
# ============================================================

def display_report_controls():

    display(

        HTML(

            "<h2 class='section-title'>📅 Report Period</h2>"

        )

    )


    display(report_type)


    display(month_dropdown)


    # DAILY CONTROL

    if report_type.value == "Daily":

        display(daily_day)


    # WEEKLY CONTROLS

    elif report_type.value == "Weekly":

        display(weekly_mode)


        if weekly_mode.value == (

            "Custom Start and End Day"

        ):

            display(

                widgets.HBox(

                    [

                        weekly_start_day,

                        weekly_end_day

                    ]

                )

            )


    # MONTHLY

    elif report_type.value == "Monthly":

        display(

            HTML(

                "<b>Monthly report: "

                "Select exactly one month.</b>"

            )

        )


    # OTHER

    elif report_type.value == "Other":

        display(

            HTML(

                "<b>Other report: "

                "Select one or more months.</b>"

            )

        )


    # YEARLY

    elif report_type.value == "Yearly":

        display(

            HTML(

                "<b>Yearly report uses all 12 "

                "monthly datasets.</b>"

            )

        )


# ============================================================
# DISPLAY ONBOARDING
# ============================================================

def display_onboarding_section():

    display(

        HTML(

            "<h2 class='section-title'>👥 Onboarding</h2>"

        )

    )


    display(

        widgets.HBox(

            [

                onboarding_option,

                view_list_button

            ]

        )

    )


    display(

        onboarding_output

    )


    update_onboarding()


# ============================================================
# DISPLAY ENGAGEMENT
# ============================================================

def display_engagement_section():

    display(

        HTML(

            "<h2 class='section-title'>📈 Engagement Graph</h2>"

        )

    )


    display(

        widgets.VBox(

            [

                engagement_mode,

                engagement_gender,

                engagement_graph_type,

                engagement_activity,

                engagement_graph_button

            ]

        )

    )


    display(

        engagement_output

    )


# ============================================================
# DISPLAY OTHER REPORT
# ============================================================

def display_other_report_section():

    display(

        HTML(

            "<h2 class='section-title'>📊 Other Report</h2>"

        )

    )


    display(

        widgets.VBox(

            [

                other_gender,

                other_activity,

                search_by,

                search_box

            ]

        )

    )


    display(

        other_report_output

    )


    generate_other_report()


# ============================================================
# MAIN DASHBOARD DISPLAY
# ============================================================

def display_dashboard(change=None):

    clear_output(wait=True)


    update_month_checkboxes()


    display(

        HTML(

            "<h1 class='dashboard-title'>"

            "📊 Attendance Dashboard"

            "</h1>"

        )

    )


    # DATASET SELECTOR

    display(file_selector)


    # GLOBAL REPORT CONTROLS

    display_report_controls()


    # MAIN SECTIONS

    display(main_section)


    if (

        main_section.value

        == "Onboarding"

    ):

        display_onboarding_section()


    elif (

        main_section.value

        == "Engagement Graph"

    ):

        display_engagement_section()


    elif (

        main_section.value

        == "Other Report"

    ):

        display_other_report_section()


# ============================================================
# EVENT HANDLERS
# ============================================================

file_selector.observe(

    display_dashboard,

    names="value"

)


report_type.observe(

    display_dashboard,

    names="value"

)


main_section.observe(

    display_dashboard,

    names="value"

)


weekly_mode.observe(

    display_dashboard,

    names="value"

)


for checkbox in month_checkboxes.values():

    checkbox.observe(

        update_onboarding,

        names="value"

    )


onboarding_option.observe(

    update_onboarding,

    names="value"

)


view_list_button.on_click(

    view_onboarding_list

)


engagement_graph_button.on_click(

    draw_engagement_graph

)


engagement_gender.observe(

    draw_engagement_graph,

    names="value"

)


other_gender.observe(

    generate_other_report,

    names="value"

)


other_activity.observe(

    generate_other_report,

    names="value"

)


search_by.observe(

    generate_other_report,

    names="value"

)


search_box.observe(

    generate_other_report,

    names="value"

)


# ============================================================
# START DASHBOARD
# ============================================================

display_dashboard()

Dropdown(description='Dataset:', layout=Layout(width='500px'), options=('DIY Attendance preparation  (7).xlsx'…

Dropdown(description='Report Type:', index=2, layout=Layout(width='500px'), options=('Daily', 'Weekly', 'Month…

ToggleButtons(index=2, layout=Layout(width='100%'), options=('Onboarding', 'Engagement Graph', 'Other Report')…

Output()